# 07 — Frozen conditional-law mechanism diagnostic

**Ticket:** `C-RLSBJTS-CONDLAW-DIAG-01`
**Claim status:** `EXPLORATORY_MECHANISM_ONLY`
**Policy training / evaluation:** `NONE` — this notebook simulates markets only.

The question is narrow and descriptive: does the frozen SBJTS target law contain
lag-dependent conditional structure, \(\mu_L(H_t)=E_L[r_t\mid H_t]\), that the
empirical iid Merton/GBM comparator cannot contain by construction?

It does **not** attribute any share of the accepted RL–SBJTS performance advantage to
the lagged-return channel, does not isolate a pure jump effect, is not a confirmatory
test, and carries no external-market-validity claim. It is post-hoc mechanism evidence
supporting an already accepted, domain-scoped comparator result.

---

## What is measured

For generated market paths only, pairs \((r_{t-1}, r_t)\) are collected for every
decision time after the first lag is available, never across a path boundary. Lagged
returns are standardised with the **frozen empirical Merton one-step calibration**,
\(z_{t-1}=(r_{t-1}-m_1)/\sqrt{v_1}\), and the same standardisation and the same fixed
bins are applied to **both** laws:

```text
(-inf,-2], (-2,-1], (-1,-0.5], (-0.5,0], (0,0.5], (0.5,1], (1,2], (2,inf)
```

Because both laws are binned on the Merton scale, bin occupancy differs between them
whenever their one-step dispersions differ. That is expected and is not itself evidence
of conditional structure; only the within-bin response is.

---

## Two run modes, never mixed

```python
RUN_MODE = "SMOKE"      # Claude may execute: tiny frozen-engine fixture, CPU, smoke/
RUN_MODE = "RESEARCH"   # user executes on the paid Colab NVIDIA T4, research/
```

`RESEARCH` hard-requires a CUDA NVIDIA T4, with no CPU fallback. That is a
*scientific* requirement here, not a speed one: the accepted comparator lineage ran the
frozen engine under `TORCH_CUDA_FLOAT32_BATCHED`, and characterising the same law under
a different numerical backend would not be the same measurement.

**No actor or critic is constructed anywhere in this notebook.** A static AST gate
(check `S8`) enforces it, and the policy overlay only reads comparator CSVs that are
already accepted evidence in the repository.


## Step 00 — run mode, sources and hardware


In [ ]:
# Step 00 — run mode, artifact resolution, hardware gate
#
# Resumable. Every artifact is located by CONTENT, never by a bare filename: a
# candidate is accepted only when its SHA-256 equals the pinned digest.

RUN_MODE = "SMOKE"          # <-- set to "RESEARCH" for the paid Colab T4 run
ALLOW_NON_T4 = False        # only with a written PMO authorisation; never enables CPU
ALLOW_NON_T4_REASON = None

import base64, hashlib, os, sys, json, shutil

TICKET = "C-RLSBJTS-CONDLAW-DIAG-01"
PROTOCOL_ID = "c9ef65485a49d40356f3bbb02d491c4b73fcc9ebf0a22f02f64ab87e04a590d4"

PINNED = {
    "03_RL_SBJTS_RESEARCH_GPU_HYBRID_v1_8.ipynb":
        "344956031d9e89763370a020d92ec54a669a7cc400e3d2b674de613897c02129",
    "frozen_market_snapshot_U1_BASELINE_4.npz":
        "7e817762849118fc3abf8d4cf98ad8d65d921fa49cb0d1b3bb34d884b73c5b4a",
    "BASE4_05A_FINAL_BUNDLE.zip":
        "77aaf6b2ccdd98d446120b72adaddf01b8819ada12b406a3bc39e073d129ca43",
    "05B_BASE4_SCIENTIFIC_EXPERIMENT_GPU_RESEARCH_v2_0.ipynb":
        "7bb73be0ddb5ad52534e6d2bdf8829fc2603394188d39f0c337b618fedf97657",
    # Frozen Base 4 RESEARCH_GPU outputs, consumed read-only and never rewritten here.
    # Pinned by content because files of the same name also exist in the SMOKE and
    # TORCHCPU output folders and a name-only match could silently pick one of those.
    "policies.npz":
        "34c39a30feb29391684bab03e4cbb0a3ebedd448af645253265413d8406fa294",
    "training_attempts.csv":
        "b56505a2d8d0973e821f0a81c60cfaaf96613c1e72bc3c11e44828f9332729c2",
}
# The frozen Base 4 target-holdout evaluation ledger. Its TT rows ARE the SBJTS arm of
# the comparator: they are read, never regenerated. It is not content-pinned here
# because the research copy is ~31 MB and grew incrementally during the Base 4 run;
# U1 is what establishes that the rows in it are the rows this code reproduces.
FROZEN_EVAL_LEDGER_NAME = "evaluation_results_partial.csv"

SEARCH_ROOTS = [
    "/content/drive/MyDrive/sbjts_rst",
    "/content/drive/MyDrive/sbjts_rst/base4_05b_live/research_outputs_GPU",
    "/content/comparator_inputs",
    os.environ.get("MERTONCOMP_INPUT_DIR", ""),
]
try:
    from google.colab import drive as _gd
    if not os.path.isdir("/content/drive/MyDrive"):
        _gd.mount("/content/drive")
except Exception as _e:
    print("[RUNTIME] Drive not mounted:", type(_e).__name__, _e)


def _sha(p, chunk=1 << 20):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()


def resolve(name, expected=None, largest_if_ambiguous=False):
    hits = []
    for root in SEARCH_ROOTS:
        if not root or not os.path.isdir(root):
            continue
        for cur, _d, files in os.walk(root):
            if name in files:
                hits.append(os.path.join(cur, name))
    if expected is not None:
        ok = sorted(p for p in hits if _sha(p) == expected)
        if not ok:
            raise RuntimeError(
                f"ARTIFACT_NOT_RESOLVED_BY_CONTENT: {name}; expected sha256 {expected}; "
                f"candidates {hits}")
        return ok[0]
    if not hits:
        raise RuntimeError(f"ARTIFACT_ABSENT: {name}")
    if largest_if_ambiguous and len(hits) > 1:
        # the research ledger is the large one; the smoke/TORCHCPU copies are tiny
        return max(hits, key=os.path.getsize)
    return sorted(hits)[0]


WORK = os.environ.get("MERTONCOMP_WORK",
                      "/content/drive/MyDrive/sbjts_rst/merton_comparator_v1")
if not os.path.isdir(os.path.dirname(WORK)):
    WORK = "/content/merton_comparator_v1"
SP = os.path.join(WORK, "inputs")
EVIDENCE_ROOT = os.path.join(WORK, "evidence")
SRC_DIR = os.path.join(WORK, "comparator_src")
for d in (SP, EVIDENCE_ROOT, SRC_DIR):
    os.makedirs(d, exist_ok=True)

STAGE = {
    "03_RL_SBJTS_RESEARCH_GPU_HYBRID_v1_8.ipynb":
        "03_RL_SBJTS_RESEARCH_GPU_HYBRID_v1_8__driveA.ipynb",
    "frozen_market_snapshot_U1_BASELINE_4.npz":
        "frozen_market_snapshot_U1_BASELINE_4.npz",
    "BASE4_05A_FINAL_BUNDLE.zip": "BASE4_05A_FINAL_BUNDLE.zip",
    "05B_BASE4_SCIENTIFIC_EXPERIMENT_GPU_RESEARCH_v2_0.ipynb": "05B_BASE4_v2_0.ipynb",
    "policies.npz": "base4_policies.npz",
    "training_attempts.csv": "base4_training_attempts.csv",
}
for name, local in STAGE.items():
    dst = os.path.join(SP, local)
    if not os.path.exists(dst):
        shutil.copyfile(resolve(name, PINNED.get(name)), dst)
    if name in PINNED and _sha(dst) != PINNED[name]:
        raise RuntimeError(f"STAGED_ARTIFACT_HASH_MISMATCH: {local}")

FROZEN_TT_LEDGER = os.path.join(SP, "base4_evaluation_results_frozen.csv")
if not os.path.exists(FROZEN_TT_LEDGER):
    shutil.copyfile(resolve(FROZEN_EVAL_LEDGER_NAME, largest_if_ambiguous=True),
                    FROZEN_TT_LEDGER)
print(f"[SOURCES] staged into {SP}; frozen TT ledger "
      f"{os.path.getsize(FROZEN_TT_LEDGER)/1e6:.1f} MB")


## Run-mode, hardware and namespace contract

Two modes that can never be confused: the output root is derived from the mode, RESEARCH refuses to start without a verified CUDA T4, and SMOKE refuses to write under the research namespace.


In [ ]:
_SRC_COMPARATOR_CONFIG_B64 = (
    "IiIiClJ1bi1tb2RlLCBoYXJkd2FyZSBhbmQgbmFtZXNwYWNlIGNvbnRyYWN0IGZvciBDLVJMU0JK"
    "VFMtTUVSVE9OLUNPTVAtMDEuCgpUd28gbW9kZXMsIG5ldmVyIG1peGVkOgoKICAgIFJVTl9NT0RF"
    "ID0gIlNNT0tFIiAgICAgQ2xhdWRlIG1heSBleGVjdXRlLiBUaW55IGJ1ZGdldHMsIHNlcGFyYXRl"
    "IG91dHB1dCBuYW1lc3BhY2UsCiAgICAgICAgICAgICAgICAgICAgICAgICAgIENQVSBwZXJtaXR0"
    "ZWQsIGV2ZXJ5IGFydGlmYWN0IHN0YW1wZWQgU01PS0VfRVZJREVOQ0UuCiAgICBSVU5fTU9ERSA9"
    "ICJSRVNFQVJDSCIgIFVzZXIgZXhlY3V0ZXMgb24gdGhlIHBhaWQgQ29sYWIgTlZJRElBIFQ0LiBG"
    "cm96ZW4gYnVkZ2V0cywKICAgICAgICAgICAgICAgICAgICAgICAgICAgQ1VEQSBmbG9hdDMyIGJh"
    "dGNoZWQgZW5naW5lLCBoYXJkIGhhcmR3YXJlIGdhdGUsIG5vIENQVQogICAgICAgICAgICAgICAg"
    "ICAgICAgICAgICBmYWxsYmFjayBvZiBhbnkga2luZC4KClRoZSBzZXBhcmF0aW9uIGlzIGVuZm9y"
    "Y2VkLCBub3QgbWVyZWx5IGRvY3VtZW50ZWQ6IHRoZSBvdXRwdXQgcm9vdCBpcyBkZXJpdmVkIGZy"
    "b20gdGhlCm1vZGUsIGEgcmVzZWFyY2ggc3RhZ2UgcmVmdXNlcyB0byBzdGFydCB3aXRob3V0IGEg"
    "dmVyaWZpZWQgQ1VEQSBUNCwgYW5kIGEgc21va2Ugc3RhZ2UKcmVmdXNlcyB0byB3cml0ZSBhbnl3"
    "aGVyZSB1bmRlciB0aGUgcmVzZWFyY2ggbmFtZXNwYWNlLgoiIiIKaW1wb3J0IGRhdGV0aW1lCmlt"
    "cG9ydCBqc29uCmltcG9ydCBvcwoKUlVOX01PREVTID0gKCJTTU9LRSIsICJSRVNFQVJDSCIpCgpQ"
    "Uk9GSUxFUyA9IHsKICAgICJTTU9LRSI6IGRpY3QoCiAgICAgICAgcnVuX21vZGU9IlNNT0tFIiwK"
    "ICAgICAgICBldmlkZW5jZV9jbGFzcz0iU01PS0VfRVZJREVOQ0UiLAogICAgICAgIGlzX3NjaWVu"
    "dGlmaWNfZXZpZGVuY2U9RmFsc2UsCiAgICAgICAgb3V0cHV0c19zdWJkaXI9InNtb2tlIiwKICAg"
    "ICAgICAjIFRpbnkgYnVkZ2V0czogZW5vdWdoIHRvIHByb3ZlIHRoZSBwaXBlbGluZSBydW5zLCBy"
    "ZXN1bWVzIGFuZCB3cml0ZXMuCiAgICAgICAgIyBgZXZhbF9wYXRoc2AgYWxvbmUgaXMgaGVsZCBh"
    "dCB0aGUgZnJvemVuIDYwMCBzbyB0aGF0IHRoZSBqb2luIGFnYWluc3QgdGhlCiAgICAgICAgIyBm"
    "cm96ZW4gQmFzZSA0IFRUIGxlZGdlciBpcyBkaW1lbnNpb25hbGx5IGZhaXRoZnVsOyB0aGUgc21v"
    "a2Ugcm93cyBhcmUgc3RpbGwKICAgICAgICAjIHNjaWVudGlmaWNhbGx5IG1lYW5pbmdsZXNzIGJl"
    "Y2F1c2UgdHJhaW5pbmcgaXMgMyB1cGRhdGVzIG9uIDIgcmVwbGljYXRpb25zLgogICAgICAgIHRy"
    "YWluX3BhdGhzPTMyLCBldmFsX3BhdGhzPTYwMCwgdXBkYXRlcz0zLAogICAgICAgIG5fcmVwbGlj"
    "YXRpb25zPTIsIGhvbGRvdXRfZW52X3N0cmVhbXM9MSwgZXZhbF9zZWVkcz0yLAogICAgICAgIGJh"
    "Y2tlbmQ9IlRPUkNIX0NQVV9GTE9BVDMyX0JBVENIRUQiLCBlbmdpbmVfZGV2aWNlPSJjcHUiLAog"
    "ICAgICAgIHJlcXVpcmVzX2N1ZGE9RmFsc2UsIHJlcXVpcmVzX3Q0PUZhbHNlLAogICAgKSwKICAg"
    "ICJSRVNFQVJDSCI6IGRpY3QoCiAgICAgICAgcnVuX21vZGU9IlJFU0VBUkNIIiwKICAgICAgICBl"
    "dmlkZW5jZV9jbGFzcz0iVVNFUl9DT0xBQl9SRVNFQVJDSF9FVklERU5DRSIsCiAgICAgICAgaXNf"
    "c2NpZW50aWZpY19ldmlkZW5jZT1UcnVlLAogICAgICAgIG91dHB1dHNfc3ViZGlyPSJyZXNlYXJj"
    "aCIsCiAgICAgICAgIyBmcm96ZW4gQmFzZSA0IGJ1ZGdldHM7IG5vbmUgb2YgdGhlc2UgbWF5IGJl"
    "IGVkaXRlZAogICAgICAgIHRyYWluX3BhdGhzPTUxMiwgZXZhbF9wYXRocz02MDAsIHVwZGF0ZXM9"
    "NDAwLAogICAgICAgIG5fcmVwbGljYXRpb25zPTQwLCBob2xkb3V0X2Vudl9zdHJlYW1zPTIwLCBl"
    "dmFsX3NlZWRzPTE1LAogICAgICAgIGJhY2tlbmQ9IlRPUkNIX0NVREFfRkxPQVQzMl9CQVRDSEVE"
    "IiwgZW5naW5lX2RldmljZT0iY3VkYTowIiwKICAgICAgICByZXF1aXJlc19jdWRhPVRydWUsIHJl"
    "cXVpcmVzX3Q0PVRydWUsCiAgICApLAp9CgojIFByZWRlY2xhcmVkIEJhc2UgNCByZXByb2R1Y3Rp"
    "b24gc3Vic2V0LiBGaXhlZCBoZXJlLCBiZWZvcmUgYW55IGNvbXBhcmlzb24gaXMgcnVuLCBhbmQK"
    "IyBkZWxpYmVyYXRlbHkgc3Bhbm5pbmcgdHdvIGhvbGRvdXQgc3RyZWFtcyBhcyB0aGUgdGlja2V0"
    "IHJlcXVpcmVzLgpSRVBST0RVQ1RJT05fU1VCU0VUID0gewogICAgImpvaW50X3RyYWluaW5nX3Jl"
    "cGxpY2F0aW9ucyI6ICgwLCAxKSwKICAgICJjb25zdHJhaW50cyI6ICgiTE9OR19PTkxZX0ZVTEwi"
    "LCAiTE9OR19PTkxZX0NBUDUwIiksCiAgICAiaG9sZG91dF9lbnZfc3RyZWFtcyI6ICgwLCAxKSwK"
    "ICAgICJldmFsX3NlZWRzIjogKDAsIDEpLAogICAgImNlbGxfY29kZSI6ICJUVCIsCiAgICAibl9y"
    "b3dzX2V4cGVjdGVkIjogMiAqIDIgKiAyICogMiwKICAgICJwcmVkZWNsYXJlZCI6IFRydWUsCiAg"
    "ICAicmF0aW9uYWxlIjogInRoZSBzbWFsbGVzdCBzdWJzZXQgdGhhdCBleGVyY2lzZXMgYm90aCBj"
    "b25zdHJhaW50IHN0cmF0YSwgdHdvICIKICAgICAgICAgICAgICAgICAiZGlzdGluY3QgaG9sZG91"
    "dCBtYXJrZXQgc3RyZWFtcyBhbmQgdHdvIGRpc3RpbmN0IGV2YWx1YXRpb24gYWN0aW9uICIKICAg"
    "ICAgICAgICAgICAgICAic2VlZHMsIHdoaWNoIGlzIHdoYXQgbWFrZXMgdGhlIGdhdGUgc2Vuc2l0"
    "aXZlIHRvIGEgc2VlZC1uYW1lc3BhY2UgIgogICAgICAgICAgICAgICAgICJvciBtYXJrZXQtZ2Vu"
    "ZXJhdG9yIGRlZmVjdCByYXRoZXIgdGhhbiBvbmx5IHRvIGFyaXRobWV0aWMgZHJpZnQiLAp9CgpF"
    "WFBFQ1RFRF9UNF9TVUJTVFJJTkdTID0gKCJUNCIsKQoKCmRlZiB1dGMoKToKICAgIHJldHVybiBk"
    "YXRldGltZS5kYXRldGltZS5ub3coZGF0ZXRpbWUudGltZXpvbmUudXRjKS5pc29mb3JtYXQoKQoK"
    "CmRlZiBwcm9maWxlKHJ1bl9tb2RlKToKICAgIGlmIHJ1bl9tb2RlIG5vdCBpbiBSVU5fTU9ERVM6"
    "CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYiVU5LTk9XTl9SVU5fTU9ERToge3J1bl9tb2Rl"
    "IXJ9OyBleHBlY3RlZCBvbmUgb2Yge1JVTl9NT0RFU30iKQogICAgcmV0dXJuIGRpY3QoUFJPRklM"
    "RVNbcnVuX21vZGVdKQoKCmRlZiBvdXRwdXRfcm9vdChldmlkZW5jZV9yb290LCBydW5fbW9kZSk6"
    "CiAgICAiIiJTTU9LRSBhbmQgUkVTRUFSQ0ggY2FuIG5ldmVyIHJlc29sdmUgdG8gdGhlIHNhbWUg"
    "ZGlyZWN0b3J5LiIiIgogICAgcCA9IHByb2ZpbGUocnVuX21vZGUpCiAgICByb290ID0gb3MucGF0"
    "aC5qb2luKGV2aWRlbmNlX3Jvb3QsIHBbIm91dHB1dHNfc3ViZGlyIl0pCiAgICBvcy5tYWtlZGly"
    "cyhyb290LCBleGlzdF9vaz1UcnVlKQogICAgcmV0dXJuIHJvb3QKCgpkZWYgYXNzZXJ0X25hbWVz"
    "cGFjZV9pc29sYXRpb24ocGF0aCwgcnVuX21vZGUsIGV2aWRlbmNlX3Jvb3QpOgogICAgIiIiUmVm"
    "dXNlIHRvIHdyaXRlIGEgc21va2UgYXJ0aWZhY3QgaW50byB0aGUgcmVzZWFyY2ggbmFtZXNwYWNl"
    "LCBvciB2aWNlIHZlcnNhLiIiIgogICAgcCA9IG9zLnBhdGguYWJzcGF0aChwYXRoKQogICAgc21v"
    "a2UgPSBvcy5wYXRoLmFic3BhdGgob3MucGF0aC5qb2luKGV2aWRlbmNlX3Jvb3QsICJzbW9rZSIp"
    "KQogICAgcmVzZWFyY2ggPSBvcy5wYXRoLmFic3BhdGgob3MucGF0aC5qb2luKGV2aWRlbmNlX3Jv"
    "b3QsICJyZXNlYXJjaCIpKQogICAgaWYgcnVuX21vZGUgPT0gIlNNT0tFIiBhbmQgcC5zdGFydHN3"
    "aXRoKHJlc2VhcmNoKToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJTTU9LRV9XUklURV9J"
    "TlRPX1JFU0VBUkNIX05BTUVTUEFDRV9GT1JCSURERU46IHtwfSIpCiAgICBpZiBydW5fbW9kZSA9"
    "PSAiUkVTRUFSQ0giIGFuZCBwLnN0YXJ0c3dpdGgoc21va2UpOgogICAgICAgIHJhaXNlIFJ1bnRp"
    "bWVFcnJvcihmIlJFU0VBUkNIX1dSSVRFX0lOVE9fU01PS0VfTkFNRVNQQUNFX0ZPUkJJRERFTjog"
    "e3B9IikKICAgIHJldHVybiBUcnVlCgoKZGVmIGhhcmR3YXJlX21hbmlmZXN0KCk6CiAgICAiIiJF"
    "dmVyeXRoaW5nIFBNTyBuZWVkcyB0byBjb25maXJtIHRoZSBydW4gcmVhbGx5IHVzZWQgdGhlIHJl"
    "bnRlZCBUNC4iIiIKICAgIG1hbiA9IHsicmVjb3JkZWRfYXRfdXRjIjogdXRjKCksICJ0b3JjaF9h"
    "dmFpbGFibGUiOiBGYWxzZSwKICAgICAgICAgICAiY3VkYV9hdmFpbGFibGUiOiBGYWxzZSwgImRl"
    "dmljZV9uYW1lIjogTm9uZSwgImRldmljZV9pbmRleCI6IE5vbmUsCiAgICAgICAgICAgImN1ZGFf"
    "cnVudGltZV92ZXJzaW9uIjogTm9uZSwgImN1ZG5uX3ZlcnNpb24iOiBOb25lLAogICAgICAgICAg"
    "ICJ0b3JjaF92ZXJzaW9uIjogTm9uZSwgInRvdGFsX21lbW9yeV9tYiI6IE5vbmUsCiAgICAgICAg"
    "ICAgImNhcGFiaWxpdHkiOiBOb25lLCAiaXNfdDQiOiBGYWxzZX0KICAgIHRyeToKICAgICAgICBp"
    "bXBvcnQgdG9yY2gKICAgIGV4Y2VwdCBJbXBvcnRFcnJvciBhcyBleGM6CiAgICAgICAgbWFuWyJp"
    "bXBvcnRfZXJyb3IiXSA9IGYie3R5cGUoZXhjKS5fX25hbWVfX306IHtleGN9IgogICAgICAgIHJl"
    "dHVybiBtYW4KICAgIG1hblsidG9yY2hfYXZhaWxhYmxlIl0gPSBUcnVlCiAgICBtYW5bInRvcmNo"
    "X3ZlcnNpb24iXSA9IHRvcmNoLl9fdmVyc2lvbl9fCiAgICBtYW5bImN1ZGFfcnVudGltZV92ZXJz"
    "aW9uIl0gPSBnZXRhdHRyKHRvcmNoLnZlcnNpb24sICJjdWRhIiwgTm9uZSkKICAgIHRyeToKICAg"
    "ICAgICBtYW5bImN1ZG5uX3ZlcnNpb24iXSA9IHRvcmNoLmJhY2tlbmRzLmN1ZG5uLnZlcnNpb24o"
    "KQogICAgZXhjZXB0IEV4Y2VwdGlvbjogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAg"
    "ICAgICAgICAgICAgIyBub3FhOiBCTEUwMDEKICAgICAgICBtYW5bImN1ZG5uX3ZlcnNpb24iXSA9"
    "IE5vbmUKICAgIGlmIHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCk6CiAgICAgICAgcHJvcHMgPSB0"
    "b3JjaC5jdWRhLmdldF9kZXZpY2VfcHJvcGVydGllcygwKQogICAgICAgIG5hbWUgPSB0b3JjaC5j"
    "dWRhLmdldF9kZXZpY2VfbmFtZSgwKQogICAgICAgIG1hbi51cGRhdGUoY3VkYV9hdmFpbGFibGU9"
    "VHJ1ZSwgZGV2aWNlX2luZGV4PTAsIGRldmljZV9uYW1lPW5hbWUsCiAgICAgICAgICAgICAgICAg"
    "ICB0b3RhbF9tZW1vcnlfbWI9aW50KHByb3BzLnRvdGFsX21lbW9yeSAvIDIgKiogMjApLAogICAg"
    "ICAgICAgICAgICAgICAgY2FwYWJpbGl0eT1mIntwcm9wcy5tYWpvcn0ue3Byb3BzLm1pbm9yfSIs"
    "CiAgICAgICAgICAgICAgICAgICBpc190ND1hbnkocyBpbiBuYW1lLnVwcGVyKCkgZm9yIHMgaW4g"
    "RVhQRUNURURfVDRfU1VCU1RSSU5HUykpCiAgICByZXR1cm4gbWFuCgoKZGVmIHJlcXVpcmVfcmVz"
    "ZWFyY2hfaGFyZHdhcmUoYWxsb3dfbm9uX3Q0PUZhbHNlLCBhbGxvd19ub25fdDRfcmVhc29uPU5v"
    "bmUpOgogICAgIiIiSGFyZCBnYXRlIGZvciBSVU5fTU9ERT0nUkVTRUFSQ0gnLiBUaGVyZSBpcyBu"
    "byBDUFUgZmFsbGJhY2sgcGF0aC4KCiAgICBgYWxsb3dfbm9uX3Q0YCBleGlzdHMgb25seSBzbyBQ"
    "TU8gY2FuIGF1dGhvcmlzZSBhIGRpZmZlcmVudCBDVURBIGRldmljZSBpbiB3cml0aW5nOwogICAg"
    "aXQgY2Fubm90IGJlIHVzZWQgdG8gcnVuIG9uIENQVSwgYW5kIHRoZSBvdmVycmlkZSBhbmQgaXRz"
    "IHJlYXNvbiBhcmUgcmVjb3JkZWQgaW4KICAgIHRoZSBtYW5pZmVzdCB0aGF0IFBNTyBhdWRpdHMu"
    "CiAgICAiIiIKICAgIG1hbiA9IGhhcmR3YXJlX21hbmlmZXN0KCkKICAgIGlmIG5vdCBtYW5bInRv"
    "cmNoX2F2YWlsYWJsZSJdOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigKICAgICAgICAgICAg"
    "IlJFU0VBUkNIX01PREVfUkVRVUlSRVNfVE9SQ0g6IFB5VG9yY2ggaXMgbm90IGltcG9ydGFibGUu"
    "IFJFU0VBUkNIIG1vZGUgIgogICAgICAgICAgICAiaGFzIG5vIENQVSBvciBOdW1QeSBmYWxsYmFj"
    "ayBieSBkZXNpZ24uIikKICAgIGlmIG5vdCBtYW5bImN1ZGFfYXZhaWxhYmxlIl06CiAgICAgICAg"
    "cmFpc2UgUnVudGltZUVycm9yKAogICAgICAgICAgICAiUkVTRUFSQ0hfTU9ERV9SRVFVSVJFU19D"
    "VURBOiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpIGlzIEZhbHNlLiBUaGUgIgogICAgICAgICAg"
    "ICAiZnJvemVuIEJhc2UgNCBudW1lcmljYWwgY29udHJhY3QgaXMgVE9SQ0hfQ1VEQV9GTE9BVDMy"
    "X0JBVENIRUQgYW5kIHRoaXMgIgogICAgICAgICAgICAidGlja2V0IGZvcmJpZHMgYSBzaWxlbnQg"
    "Q1BVIGZhbGxiYWNrLiBJbiBDb2xhYiBjaG9vc2UgIgogICAgICAgICAgICAiUnVudGltZSA+IENo"
    "YW5nZSBydW50aW1lIHR5cGUgPiBUNCBHUFUgYW5kIHJlLXJ1biBmcm9tIHRoZSB0b3AuIikKICAg"
    "IGlmIG5vdCBtYW5bImlzX3Q0Il0gYW5kIG5vdCBhbGxvd19ub25fdDQ6CiAgICAgICAgcmFpc2Ug"
    "UnVudGltZUVycm9yKAogICAgICAgICAgICBmIlJFU0VBUkNIX01PREVfRVhQRUNUU19OVklESUFf"
    "VDQ6IGFsbG9jYXRlZCBkZXZpY2UgaXMgIgogICAgICAgICAgICBmInttYW5bJ2RldmljZV9uYW1l"
    "J10hcn0uIFRoZSByZXNlYXJjaCBwbGFuLCBiYXRjaCBzaXplcyBhbmQgcnVudGltZSAiCiAgICAg"
    "ICAgICAgICJlc3RpbWF0ZXMgYXJlIHdyaXR0ZW4gZm9yIHRoZSByZW50ZWQgVDQuIFJ1bm5pbmcg"
    "b24gYW5vdGhlciBHUFUgbmVlZHMgYW4gIgogICAgICAgICAgICAiZXhwbGljaXQgUE1PIGF1dGhv"
    "cmlzYXRpb247IHNldCBhbGxvd19ub25fdDQ9VHJ1ZSB3aXRoIGEgd3JpdHRlbiByZWFzb24gIgog"
    "ICAgICAgICAgICAib25seSBhZnRlciBQTU8gcmVjb3JkcyB0aGF0IGRlY2lzaW9uLiIpCiAgICBt"
    "YW5bImFsbG93X25vbl90NF9vdmVycmlkZSJdID0gYm9vbChhbGxvd19ub25fdDQpCiAgICBtYW5b"
    "ImFsbG93X25vbl90NF9yZWFzb24iXSA9IGFsbG93X25vbl90NF9yZWFzb24KICAgIG1hblsiZ2F0"
    "ZSJdID0gIlJFU0VBUkNIX0hBUkRXQVJFX0dBVEVfUEFTUyIKICAgIHJldHVybiBtYW4KCgpkZWYg"
    "cmVzb2x2ZV9iYWNrZW5kKHJ1bl9tb2RlLCBhbGxvd19ub25fdDQ9RmFsc2UsIGFsbG93X25vbl90"
    "NF9yZWFzb249Tm9uZSk6CiAgICAiIiJSZXR1cm4gKGJhY2tlbmQsIGRldmljZSwgaGFyZHdhcmUg"
    "bWFuaWZlc3QpIGZvciB0aGUgbW9kZS4gUkVTRUFSQ0ggbmV2ZXIgZmFsbHMgYmFjay4iIiIKICAg"
    "IHAgPSBwcm9maWxlKHJ1bl9tb2RlKQogICAgaWYgcnVuX21vZGUgPT0gIlJFU0VBUkNIIjoKICAg"
    "ICAgICBtYW4gPSByZXF1aXJlX3Jlc2VhcmNoX2hhcmR3YXJlKGFsbG93X25vbl90NCwgYWxsb3df"
    "bm9uX3Q0X3JlYXNvbikKICAgICAgICByZXR1cm4gcFsiYmFja2VuZCJdLCBwWyJlbmdpbmVfZGV2"
    "aWNlIl0sIG1hbgogICAgbWFuID0gaGFyZHdhcmVfbWFuaWZlc3QoKQogICAgbWFuWyJnYXRlIl0g"
    "PSAiU01PS0VfTk9fSEFSRFdBUkVfUkVRVUlSRU1FTlQiCiAgICAjIFNNT0tFIHN0YXlzIG9uIENQ"
    "VSBldmVuIHdoZW4gYSBHUFUgaGFwcGVucyB0byBiZSBwcmVzZW50LCBzbyB0aGF0IGEgc21va2Ug"
    "cnVuIGNhbgogICAgIyBuZXZlciBiZSBtaXN0YWtlbiBmb3IsIG9yIHNpbGVudGx5IG1lcmdlZCB3"
    "aXRoLCBhIHJlc2VhcmNoIHJ1bi4KICAgIG1hblsic21va2VfZm9yY2VkX2NwdSJdID0gVHJ1ZQog"
    "ICAgcmV0dXJuIHBbImJhY2tlbmQiXSwgcFsiZW5naW5lX2RldmljZSJdLCBtYW4KCgpkZWYgc3Rh"
    "bXAocGF5bG9hZCwgcnVuX21vZGUsIHN0YWdlKToKICAgICIiIkV2ZXJ5IGFydGlmYWN0IGNhcnJp"
    "ZXMgaXRzIG1vZGUsIGV2aWRlbmNlIGNsYXNzIGFuZCBhIGNsYWltLXN0YXR1cyByZW1pbmRlci4i"
    "IiIKICAgIHAgPSBwcm9maWxlKHJ1bl9tb2RlKQogICAgb3V0ID0gewogICAgICAgICJ0aWNrZXQi"
    "OiAiQy1STFNCSlRTLU1FUlRPTi1DT01QLTAxIiwKICAgICAgICAic3RhZ2UiOiBzdGFnZSwKICAg"
    "ICAgICAicnVuX21vZGUiOiBydW5fbW9kZSwKICAgICAgICAiZXZpZGVuY2VfY2xhc3MiOiBwWyJl"
    "dmlkZW5jZV9jbGFzcyJdLAogICAgICAgICJpc19zY2llbnRpZmljX2V2aWRlbmNlIjogcFsiaXNf"
    "c2NpZW50aWZpY19ldmlkZW5jZSJdLAogICAgICAgICJnZW5lcmF0ZWRfYXRfdXRjIjogdXRjKCks"
    "CiAgICB9CiAgICBpZiBydW5fbW9kZSA9PSAiU01PS0UiOgogICAgICAgIG91dFsiY2xhaW1fc3Rh"
    "dHVzIl0gPSAoIlNNT0tFX0VWSURFTkNFIOKAlCBleGVjdXRpb24gcHJvb2Ygb25seS4gVGhpcyBh"
    "cnRpZmFjdCBpcyAiCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAibm90IGEgc2NpZW50"
    "aWZpYyByZXN1bHQgYW5kIG11c3QgbmV2ZXIgYmUgcmVwb3J0ZWQgYXMgIgogICAgICAgICAgICAg"
    "ICAgICAgICAgICAgICAgICAgImNvbXBhcmF0b3IgZXZpZGVuY2UuIikKICAgIG91dC51cGRhdGUo"
    "cGF5bG9hZCkKICAgIHJldHVybiBvdXQKCgpkZWYgd3JpdGVfanNvbihwYXRoLCBwYXlsb2FkLCBy"
    "dW5fbW9kZSwgc3RhZ2UsIGV2aWRlbmNlX3Jvb3QpOgogICAgYXNzZXJ0X25hbWVzcGFjZV9pc29s"
    "YXRpb24ocGF0aCwgcnVuX21vZGUsIGV2aWRlbmNlX3Jvb3QpCiAgICBvcy5tYWtlZGlycyhvcy5w"
    "YXRoLmRpcm5hbWUocGF0aCksIGV4aXN0X29rPVRydWUpCiAgICB0bXAgPSBwYXRoICsgIi50bXAi"
    "CiAgICB3aXRoIG9wZW4odG1wLCAidyIsIGVuY29kaW5nPSJ1dGYtOCIpIGFzIGY6CiAgICAgICAg"
    "anNvbi5kdW1wKHN0YW1wKHBheWxvYWQsIHJ1bl9tb2RlLCBzdGFnZSksIGYsIGluZGVudD0yKQog"
    "ICAgICAgIGYuZmx1c2goKQogICAgICAgIG9zLmZzeW5jKGYuZmlsZW5vKCkpCiAgICBqc29uLmxv"
    "YWQob3Blbih0bXAsIGVuY29kaW5nPSJ1dGYtOCIpKQogICAgb3MucmVwbGFjZSh0bXAsIHBhdGgp"
    "CiAgICByZXR1cm4gcGF0aAo="
)
_SRC_COMPARATOR_CONFIG = base64.b64decode(_SRC_COMPARATOR_CONFIG_B64).decode()
open(os.path.join(SRC_DIR, "comparator_config.py"), "w").write(_SRC_COMPARATOR_CONFIG)
print('comparator_config.py staged', len(_SRC_COMPARATOR_CONFIG), 'chars')


## Frozen-source loader

Rebuilds the Base 3 engine/learner namespace and the Base 4 protocol glue from verified artifact bytes. No frozen mathematics is re-implemented: every scientific object is executed from the verified Base 3 source text.


In [ ]:
_SRC_FROZEN_LOADER_B64 = (
    "..."
    "ID0gY3RsCiAgICAgICAgcmV0dXJuIG91dAo="
)
_SRC_FROZEN_LOADER = base64.b64decode(_SRC_FROZEN_LOADER_B64).decode()
open(os.path.join(SRC_DIR, "frozen_loader.py"), "w").write(_SRC_FROZEN_LOADER)
print('frozen_loader.py staged', len(_SRC_FROZEN_LOADER), 'chars')


## Merton/GBM arm

Section 6.1 empirical GBM calibration and its predeclared Monte Carlo moment test, the GBM market batch built through the frozen `build_market_batch`, the Base 4 training body with the market law swapped, and the Section 6.2 analytic exploratory Merton policy at both exploration conventions.


In [ ]:
_SRC_MERTON_ARM_B64 = (
    "..."
)
_SRC_MERTON_ARM = base64.b64decode(_SRC_MERTON_ARM_B64).decode()
open(os.path.join(SRC_DIR, "merton_arm.py"), "w").write(_SRC_MERTON_ARM)
print('merton_arm.py staged', len(_SRC_MERTON_ARM), 'chars')


## Module — fixed binning and block sufficient statistics

One fixed bin scheme for both laws; every reported quantity is a function of per-(law, block, bin) sufficient statistics, which is what makes the run resumable and the bootstrap block-level.


In [ ]:
_SRC_BINNING_B64 = (
    "IiIiCkMtUkxTQkpUUy1DT05ETEFXLURJQUctMDEg4oCUIGZpeGVkIGJpbm5pbmcgYW5kIGJsb2Nr"
    "LWxldmVsIHN1ZmZpY2llbnQgc3RhdGlzdGljcy4KCkV2ZXJ5dGhpbmcgdGhlIGRpYWdub3N0aWMg"
    "cmVwb3J0cyBpcyBhIGZ1bmN0aW9uIG9mIHBlci0obGF3LCBibG9jaywgYmluKSBzdWZmaWNpZW50"
    "CnN0YXRpc3RpY3MuIFRoYXQgY2hvaWNlIGlzIGxvYWQgYmVhcmluZyB0aHJlZSB0aW1lcyBvdmVy"
    "OgoKICAqIGJsb2NrcyBjYW4gYmUgY2hlY2twb2ludGVkIGFuZCByZXN1bWVkIHdpdGhvdXQga2Vl"
    "cGluZyByYXcgcGFpcnM7CiAgKiBwb29sZWQgZXN0aW1hdGVzIGFyZSBFWEFDVExZIHJlY29uc3Ry"
    "dWN0aWJsZSBmcm9tIHRoZSBzdG9yZWQgYmxvY2tzLCBzbyBhIHJlc3VtZWQKICAgIHJ1biBlcXVh"
    "bHMgYW4gdW5pbnRlcnJ1cHRlZCBvbmUgYml0IGZvciBiaXQ7CiAgKiB0aGUgY2x1c3RlciBib290"
    "c3RyYXAgcmVzYW1wbGVzIGJsb2NrcyBhbmQgcmVjb21iaW5lcyBzdGF0cywgc28gdW5jZXJ0YWlu"
    "dHkgaXMgYXQKICAgIHRoZSBzaW11bGF0aW9uLWJsb2NrIGxldmVsIHJhdGhlciB0aGFuIHRoZSBy"
    "b3cgbGV2ZWwsIGFzIHRoZSB0aWNrZXQgcmVxdWlyZXMuCiIiIgppbXBvcnQgbWF0aAoKaW1wb3J0"
    "IG51bXB5IGFzIG5wCgojIFRpY2tldCBzZWN0aW9uIDQ6IG9uZSBmaXhlZCBjb21tb24gYmlubmlu"
    "ZyBzY2hlbWUsIGFwcGxpZWQgdG8gQk9USCBsYXdzLgpaX0VER0VTID0gKC1ucC5pbmYsIC0yLjAs"
    "IC0xLjAsIC0wLjUsIDAuMCwgMC41LCAxLjAsIDIuMCwgbnAuaW5mKQpCSU5fTEFCRUxTID0gKCIo"
    "LWluZiwtMl0iLCAiKC0yLC0xXSIsICIoLTEsLTAuNV0iLCAiKC0wLjUsMF0iLAogICAgICAgICAg"
    "ICAgICIoMCwwLjVdIiwgIigwLjUsMV0iLCAiKDEsMl0iLCAiKDIsaW5mKSIpCk5fQklOUyA9IGxl"
    "bihCSU5fTEFCRUxTKQoKIyBGaWVsZHMgb2YgdGhlIHBlci0oYmxvY2ssIGJpbikgc3VmZmljaWVu"
    "dCBzdGF0aXN0aWMgYmxvY2suCkJJTl9GSUVMRFMgPSAoIm4iLCAic3VtX2xhZyIsICJzdW1fciIs"
    "ICJzdW1fcjIiLCAibl90YWlsIikKIyBGaWVsZHMgb2YgdGhlIHBlci1ibG9jayBnbG9iYWwgc3Rh"
    "dGlzdGljIGJsb2NrLgpHTE9CQUxfRklFTERTID0gKCJuX3BhaXJzIiwgInN1bV94IiwgInN1bV95"
    "IiwgInN1bV94eSIsICJzdW1feDIiLCAic3VtX3kyIiwKICAgICAgICAgICAgICAgICAibl9hbGwi"
    "LCAic3VtX2FsbCIsICJzdW1fYWxsMiIpCgoKZGVmIGJpbl9pbmRleCh6KToKICAgICIiIlJpZ2h0"
    "LWNsb3NlZCBiaW5zLCBzbyBhbiBvYnNlcnZhdGlvbiBvbiBhbiBlZGdlIGZhbGxzIGluIHRoZSBM"
    "T1dFUiBiaW4uCgogICAgbnAuZGlnaXRpemUgd2l0aCByaWdodD1UcnVlIHJldHVybnMgMCBmb3Ig"
    "eiA8PSAtaW5mIChpbXBvc3NpYmxlKSBhbmQgTiBmb3IKICAgIHogPiAraW5mIChpbXBvc3NpYmxl"
    "KSwgc28gdGhlIHVzYWJsZSByYW5nZSBpcyAxLi5OX0JJTlMgYW5kIHdlIHNoaWZ0IHRvIDAtYmFz"
    "ZWQuCiAgICAiIiIKICAgIGlkeCA9IG5wLmRpZ2l0aXplKG5wLmFzYXJyYXkoeiwgbnAuZmxvYXQ2"
    "NCksIG5wLmFzYXJyYXkoWl9FREdFU1sxOi0xXSwgbnAuZmxvYXQ2NCksCiAgICAgICAgICAgICAg"
    "ICAgICAgICByaWdodD1UcnVlKQogICAgcmV0dXJuIGlkeC5hc3R5cGUobnAuaW50NjQpCgoKZGVm"
    "IG1lcnRvbl90YWlsX3F1YW50aWxlKG0xLCB2MSwgYWxwaGE9MC4wNSk6CiAgICAiIiJGaXhlZCBN"
    "ZXJ0b24gb25lLXN0ZXAgYWxwaGEtcXVhbnRpbGU7IHRoZSBTQU1FIGNvbnN0YW50IGlzIHVzZWQg"
    "Zm9yIGJvdGggbGF3cy4iIiIKICAgIGZyb20gc2NpcHkuc3RhdHMgaW1wb3J0IG5vcm0KICAgIHJl"
    "dHVybiBmbG9hdChtMSArIG1hdGguc3FydCh2MSkgKiBmbG9hdChub3JtLnBwZihhbHBoYSkpKQoK"
    "CmRlZiBibG9ja19zdGF0aXN0aWNzKHJldHVybnMsIG0xLCB2MSwgcV90YWlsKToKICAgICIiIlN1"
    "ZmZpY2llbnQgc3RhdGlzdGljcyBmb3Igb25lIHNpbXVsYXRpb24gYmxvY2suCgogICAgYHJldHVy"
    "bnNgIGlzIChQLCBOKSBvZiBwZXItc3RlcCByaXNreSBsb2cgcmV0dXJucyBmb3Igb25lIGxhdyBh"
    "bmQgb25lIHNlZWQgYmxvY2suCiAgICBQYWlycyAocl97dC0xfSwgcl90KSBhcmUgdGFrZW4gZm9y"
    "IHQgPSAxIC4uIE4tMSwgaS5lLiBvbmx5IGFmdGVyIHRoZSBmaXJzdCBsYWcgaXMKICAgIGF2YWls"
    "YWJsZSwgYW5kIG5ldmVyIGFjcm9zcyBwYXRoIGJvdW5kYXJpZXMuCiAgICAiIiIKICAgIHIgPSBu"
    "cC5hc2FycmF5KHJldHVybnMsIG5wLmZsb2F0NjQpCiAgICBpZiByLm5kaW0gIT0gMiBvciByLnNo"
    "YXBlWzFdIDwgMjoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiZXhwZWN0ZWQgKFAsIE4+PTIp"
    "IHJldHVybnMsIGdvdCB7ci5zaGFwZX0iKQogICAgbGFnID0gcls6LCA6LTFdLnJlc2hhcGUoLTEp"
    "CiAgICBjdXIgPSByWzosIDE6XS5yZXNoYXBlKC0xKQogICAgeiA9IChsYWcgLSBmbG9hdChtMSkp"
    "IC8gbWF0aC5zcXJ0KGZsb2F0KHYxKSkKICAgIGIgPSBiaW5faW5kZXgoeikKCiAgICBiaW5zID0g"
    "bnAuemVyb3MoKE5fQklOUywgbGVuKEJJTl9GSUVMRFMpKSwgbnAuZmxvYXQ2NCkKICAgIHRhaWwg"
    "PSAoY3VyIDw9IGZsb2F0KHFfdGFpbCkpLmFzdHlwZShucC5mbG9hdDY0KQogICAgZm9yIGosIGFy"
    "ciBpbiBlbnVtZXJhdGUoKG5wLm9uZXNfbGlrZShjdXIpLCBsYWcsIGN1ciwgY3VyICogY3VyLCB0"
    "YWlsKSk6CiAgICAgICAgYmluc1s6LCBqXSA9IG5wLmJpbmNvdW50KGIsIHdlaWdodHM9YXJyLCBt"
    "aW5sZW5ndGg9Tl9CSU5TKQoKICAgIGFsbHIgPSByLnJlc2hhcGUoLTEpCiAgICBnbG9iID0gbnAu"
    "YXJyYXkoWwogICAgICAgIGZsb2F0KGN1ci5zaXplKSwgZmxvYXQobGFnLnN1bSgpKSwgZmxvYXQo"
    "Y3VyLnN1bSgpKSwgZmxvYXQoKGxhZyAqIGN1cikuc3VtKCkpLAogICAgICAgIGZsb2F0KChsYWcg"
    "KiBsYWcpLnN1bSgpKSwgZmxvYXQoKGN1ciAqIGN1cikuc3VtKCkpLAogICAgICAgIGZsb2F0KGFs"
    "bHIuc2l6ZSksIGZsb2F0KGFsbHIuc3VtKCkpLCBmbG9hdCgoYWxsciAqIGFsbHIpLnN1bSgpKSwK"
    "ICAgIF0sIG5wLmZsb2F0NjQpCiAgICByZXR1cm4geyJiaW5zIjogYmlucywgImdsb2JhbCI6IGds"
    "b2J9CgoKZGVmIGNvbWJpbmUoYmxvY2tzKToKICAgICIiIkV4YWN0IHBvb2xpbmcgb2YgYmxvY2sg"
    "c3RhdGlzdGljcy4gQWRkaXRpb24gaXMgYWxsIGl0IHRha2VzLCBieSBjb25zdHJ1Y3Rpb24uIiIi"
    "CiAgICBpZiBub3QgYmxvY2tzOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIm5vIGJsb2NrcyB0"
    "byBjb21iaW5lIikKICAgIGJpbnMgPSBucC56ZXJvc19saWtlKGJsb2Nrc1swXVsiYmlucyJdKQog"
    "ICAgZ2xvYiA9IG5wLnplcm9zX2xpa2UoYmxvY2tzWzBdWyJnbG9iYWwiXSkKICAgIGZvciBiIGlu"
    "IGJsb2NrczoKICAgICAgICBiaW5zICs9IGJbImJpbnMiXQogICAgICAgIGdsb2IgKz0gYlsiZ2xv"
    "YmFsIl0KICAgIHJldHVybiB7ImJpbnMiOiBiaW5zLCAiZ2xvYmFsIjogZ2xvYn0KCgpkZWYgX3Nh"
    "ZmVfZGl2KG51bSwgZGVuKToKICAgIHJldHVybiBmbG9hdChudW0gLyBkZW4pIGlmIGRlbiA+IDAg"
    "ZWxzZSBmbG9hdCgibmFuIikKCgpkZWYgZXN0aW1hdGVzKHBvb2xlZCwgbWluX249Mik6CiAgICAi"
    "IiJQb2ludCBlc3RpbWF0ZXMgZnJvbSBwb29sZWQgc3VmZmljaWVudCBzdGF0aXN0aWNzLiIiIgog"
    "ICAgYmlucywgZyA9IHBvb2xlZFsiYmlucyJdLCBwb29sZWRbImdsb2JhbCJdCiAgICBuX2FsbCwg"
    "c3VtX2FsbCwgc3VtX2FsbDIgPSBnWzZdLCBnWzddLCBnWzhdCiAgICB1bmNvbmRfbWVhbiA9IF9z"
    "YWZlX2RpdihzdW1fYWxsLCBuX2FsbCkKICAgIHVuY29uZF92YXIgPSAoX3NhZmVfZGl2KHN1bV9h"
    "bGwyIC0gc3VtX2FsbCAqIHN1bV9hbGwgLyBuX2FsbCwgbl9hbGwgLSAxLjApCiAgICAgICAgICAg"
    "ICAgICAgIGlmIG5fYWxsID4gMSBlbHNlIGZsb2F0KCJuYW4iKSkKCiAgICByb3dzID0gW10KICAg"
    "IGZvciBpIGluIHJhbmdlKE5fQklOUyk6CiAgICAgICAgbiwgc19sYWcsIHNfciwgc19yMiwgbl90"
    "YWlsID0gYmluc1tpXQogICAgICAgIG1lYW5fciA9IF9zYWZlX2RpdihzX3IsIG4pCiAgICAgICAg"
    "dmFyX3IgPSAoX3NhZmVfZGl2KHNfcjIgLSBzX3IgKiBzX3IgLyBuLCBuIC0gMS4wKSBpZiBuID49"
    "IG1pbl9uIGVsc2UgZmxvYXQoIm5hbiIpKQogICAgICAgIHJvd3MuYXBwZW5kKHsKICAgICAgICAg"
    "ICAgImJpbiI6IEJJTl9MQUJFTFNbaV0sCiAgICAgICAgICAgICJiaW5faW5kZXgiOiBpLAogICAg"
    "ICAgICAgICAibiI6IGludChuKSwKICAgICAgICAgICAgIm1lYW5fbGFnZ2VkX3JldHVybiI6IF9z"
    "YWZlX2RpdihzX2xhZywgbiksCiAgICAgICAgICAgICJjb25kaXRpb25hbF9tZWFuIjogbWVhbl9y"
    "LAogICAgICAgICAgICAiZGV2aWF0aW9uX2Zyb21fdW5jb25kaXRpb25hbF9tZWFuIjoKICAgICAg"
    "ICAgICAgICAgIChtZWFuX3IgLSB1bmNvbmRfbWVhbikgaWYgbiA+IDAgZWxzZSBmbG9hdCgibmFu"
    "IiksCiAgICAgICAgICAgICJjb25kaXRpb25hbF92YXJpYW5jZSI6IHZhcl9yLAogICAgICAgICAg"
    "ICAidGFpbF9wcm9iYWJpbGl0eSI6IF9zYWZlX2RpdihuX3RhaWwsIG4pLAogICAgICAgIH0pCgog"
    "ICAgbl9wLCBzeCwgc3ksIHN4eSwgc3gyLCBzeTIgPSBnWzBdLCBnWzFdLCBnWzJdLCBnWzNdLCBn"
    "WzRdLCBnWzVdCiAgICBkZW4gPSBuX3AgKiBzeDIgLSBzeCAqIHN4CiAgICBzbG9wZSA9IF9zYWZl"
    "X2RpdihuX3AgKiBzeHkgLSBzeCAqIHN5LCBkZW4pIGlmIGRlbiAhPSAwIGVsc2UgZmxvYXQoIm5h"
    "biIpCiAgICBjb3YgPSBfc2FmZV9kaXYoc3h5IC0gc3ggKiBzeSAvIG5fcCwgbl9wIC0gMS4wKSBp"
    "ZiBuX3AgPiAxIGVsc2UgZmxvYXQoIm5hbiIpCiAgICB2eCA9IF9zYWZlX2RpdihzeDIgLSBzeCAq"
    "IHN4IC8gbl9wLCBuX3AgLSAxLjApIGlmIG5fcCA+IDEgZWxzZSBmbG9hdCgibmFuIikKICAgIHZ5"
    "ID0gX3NhZmVfZGl2KHN5MiAtIHN5ICogc3kgLyBuX3AsIG5fcCAtIDEuMCkgaWYgbl9wID4gMSBl"
    "bHNlIGZsb2F0KCJuYW4iKQogICAgZGVub20gPSBtYXRoLnNxcnQodnggKiB2eSkgaWYgKHZ4ID4g"
    "MCBhbmQgdnkgPiAwKSBlbHNlIGZsb2F0KCJuYW4iKQogICAgcmV0dXJuIHsKICAgICAgICAiYmlu"
    "cyI6IHJvd3MsCiAgICAgICAgInVuY29uZGl0aW9uYWxfbWVhbiI6IHVuY29uZF9tZWFuLAogICAg"
    "ICAgICJ1bmNvbmRpdGlvbmFsX3ZhcmlhbmNlIjogdW5jb25kX3ZhciwKICAgICAgICAibl9yZXR1"
    "cm5zIjogaW50KG5fYWxsKSwKICAgICAgICAibl9wYWlycyI6IGludChuX3ApLAogICAgICAgICJs"
    "YWdfc2xvcGUiOiBzbG9wZSwKICAgICAgICAibGFnMV9hdXRvY292YXJpYW5jZSI6IGNvdiwKICAg"
    "ICAgICAibGFnMV9hdXRvY29ycmVsYXRpb24iOiAoY292IC8gZGVub20gaWYgZGVub20gPT0gZGVu"
    "b20gYW5kIGRlbm9tID4gMAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBlbHNlIGZs"
    "b2F0KCJuYW4iKSksCiAgICB9CgoKZGVmIF9mbGF0dGVuKGVzdCk6CiAgICAiIiJTY2FsYXIgdmVj"
    "dG9yIG9mIGV2ZXJ5IHF1YW50aXR5IHRoYXQgZ2V0cyBhIGNvbmZpZGVuY2UgaW50ZXJ2YWwuIiIi"
    "CiAgICBvdXQgPSBbXQogICAgZm9yIHJvdyBpbiBlc3RbImJpbnMiXToKICAgICAgICBvdXQgKz0g"
    "W3Jvd1siY29uZGl0aW9uYWxfbWVhbiJdLCByb3dbImRldmlhdGlvbl9mcm9tX3VuY29uZGl0aW9u"
    "YWxfbWVhbiJdLAogICAgICAgICAgICAgICAgcm93WyJjb25kaXRpb25hbF92YXJpYW5jZSJdLCBy"
    "b3dbInRhaWxfcHJvYmFiaWxpdHkiXV0KICAgIG91dCArPSBbZXN0WyJsYWdfc2xvcGUiXSwgZXN0"
    "WyJsYWcxX2F1dG9jb3JyZWxhdGlvbiJdLAogICAgICAgICAgICBlc3RbInVuY29uZGl0aW9uYWxf"
    "bWVhbiJdLCBlc3RbInVuY29uZGl0aW9uYWxfdmFyaWFuY2UiXV0KICAgIHJldHVybiBucC5hc2Fy"
    "cmF5KG91dCwgbnAuZmxvYXQ2NCkKCgpkZWYgY2x1c3Rlcl9ib290c3RyYXAoYmxvY2tzLCBuX2Jv"
    "b3Q9MjAwMCwgc2VlZD0wLCBhbHBoYT0wLjA1LCBtaW5fbj0yKToKICAgICIiIkNsdXN0ZXIgYm9v"
    "dHN0cmFwIHdpdGggdGhlIFNJTVVMQVRJT04gQkxPQ0sgYXMgdGhlIHJlc2FtcGxpbmcgdW5pdC4K"
    "CiAgICBCbG9ja3MgYXJlIHJlc2FtcGxlZCB3aXRoIHJlcGxhY2VtZW50IGFuZCB0aGVpciBzdWZm"
    "aWNpZW50IHN0YXRpc3RpY3MgcmUtcG9vbGVkLCBzbwogICAgdGhlIGludGVydmFsIHJlZmxlY3Rz"
    "IGJldHdlZW4tYmxvY2sgdmFyaWF0aW9uLiBSb3ctbGV2ZWwgaWlkIHJlc2FtcGxpbmcgd291bGQg"
    "YmFkbHkKICAgIHVuZGVyc3RhdGUgaXQsIGJlY2F1c2UgcmV0dXJucyB3aXRoaW4gYSBwYXRoIGFy"
    "ZSBkZXBlbmRlbnQgYnkgY29uc3RydWN0aW9uIHVuZGVyIHRoZQogICAgdmVyeSBsYXcgYmVpbmcg"
    "dGVzdGVkLgogICAgIiIiCiAgICBuX2Jsb2NrcyA9IGxlbihibG9ja3MpCiAgICBpZiBuX2Jsb2Nr"
    "cyA8IDI6CiAgICAgICAgcmV0dXJuIHsibl9ibG9ja3MiOiBuX2Jsb2NrcywgIm5fYm9vdCI6IDAs"
    "CiAgICAgICAgICAgICAgICAic3RhdHVzIjogIkNMVVNURVJfQk9PVFNUUkFQX1JFUVVJUkVTX0FU"
    "X0xFQVNUX1RXT19CTE9DS1MiLAogICAgICAgICAgICAgICAgImxvIjogTm9uZSwgImhpIjogTm9u"
    "ZX0KICAgIHJuZyA9IG5wLnJhbmRvbS5kZWZhdWx0X3JuZyhpbnQoc2VlZCkpCiAgICBkcmF3cyA9"
    "IFtdCiAgICBmb3IgXyBpbiByYW5nZShpbnQobl9ib290KSk6CiAgICAgICAgcGljayA9IHJuZy5p"
    "bnRlZ2VycygwLCBuX2Jsb2Nrcywgc2l6ZT1uX2Jsb2NrcykKICAgICAgICBkcmF3cy5hcHBlbmQo"
    "X2ZsYXR0ZW4oZXN0aW1hdGVzKGNvbWJpbmUoW2Jsb2Nrc1tpXSBmb3IgaSBpbiBwaWNrXSksIG1p"
    "bl9uKSkpCiAgICBEID0gbnAuYXNhcnJheShkcmF3cywgbnAuZmxvYXQ2NCkKICAgIGxvID0gbnAu"
    "bmFucGVyY2VudGlsZShELCAxMDAuMCAqIGFscGhhIC8gMi4wLCBheGlzPTApCiAgICBoaSA9IG5w"
    "Lm5hbnBlcmNlbnRpbGUoRCwgMTAwLjAgKiAoMS4wIC0gYWxwaGEgLyAyLjApLCBheGlzPTApCiAg"
    "ICByZXR1cm4geyJuX2Jsb2NrcyI6IG5fYmxvY2tzLCAibl9ib290IjogaW50KG5fYm9vdCksICJh"
    "bHBoYSI6IGFscGhhLAogICAgICAgICAgICAic3RhdHVzIjogIk9LIiwgImxvIjogbG8udG9saXN0"
    "KCksICJoaSI6IGhpLnRvbGlzdCgpLAogICAgICAgICAgICAicmVzYW1wbGluZ191bml0IjogInNp"
    "bXVsYXRpb24gc2VlZCBibG9jayJ9CgoKZGVmIGF0dGFjaF9pbnRlcnZhbHMoZXN0LCBib290KToK"
    "ICAgICIiIkF0dGFjaCB0aGUgYm9vdHN0cmFwIGludGVydmFsIHRvIGVhY2ggZXN0aW1hdGUsIGlu"
    "IF9mbGF0dGVuIG9yZGVyLiIiIgogICAgaWYgYm9vdC5nZXQoInN0YXR1cyIpICE9ICJPSyI6CiAg"
    "ICAgICAgcmV0dXJuIGVzdAogICAgbG8sIGhpLCBrID0gYm9vdFsibG8iXSwgYm9vdFsiaGkiXSwg"
    "MAogICAgZm9yIHJvdyBpbiBlc3RbImJpbnMiXToKICAgICAgICBmb3Iga2V5IGluICgiY29uZGl0"
    "aW9uYWxfbWVhbiIsICJkZXZpYXRpb25fZnJvbV91bmNvbmRpdGlvbmFsX21lYW4iLAogICAgICAg"
    "ICAgICAgICAgICAgICJjb25kaXRpb25hbF92YXJpYW5jZSIsICJ0YWlsX3Byb2JhYmlsaXR5Iik6"
    "CiAgICAgICAgICAgIHJvd1trZXkgKyAiX2xvIl0gPSBsb1trXQogICAgICAgICAgICByb3dba2V5"
    "ICsgIl9oaSJdID0gaGlba10KICAgICAgICAgICAgayArPSAxCiAgICBmb3Iga2V5IGluICgibGFn"
    "X3Nsb3BlIiwgImxhZzFfYXV0b2NvcnJlbGF0aW9uIiwKICAgICAgICAgICAgICAgICJ1bmNvbmRp"
    "dGlvbmFsX21lYW4iLCAidW5jb25kaXRpb25hbF92YXJpYW5jZSIpOgogICAgICAgIGVzdFtrZXkg"
    "KyAiX2xvIl0gPSBsb1trXQogICAgICAgIGVzdFtrZXkgKyAiX2hpIl0gPSBoaVtrXQogICAgICAg"
    "IGsgKz0gMQogICAgcmV0dXJuIGVzdAoKCmRlZiBzdXBfZmxhdG5lc3NfdGVzdChibG9ja3MsIG51"
    "bGxfdmFsdWUsIG5fYm9vdD0yMDAwLCBzZWVkPTAsIG1pbl9uPTEsIGxldmVsPTAuOTUpOgogICAg"
    "IiIiU2ltdWx0YW5lb3VzIGZsYXRuZXNzIHRlc3QgdmlhIGEgc3R1ZGVudGl6ZWQgc3VwLXN0YXRp"
    "c3RpYyBvdmVyIGJpbnMuCgogICAgV2h5IG5vdCBwZXItYmluIGludGVydmFsczogcmVhZGluZyBl"
    "aWdodCA5NSUgaW50ZXJ2YWxzIHNpbXVsdGFuZW91c2x5IHJlamVjdHMgYQogICAgZ2VudWluZWx5"
    "IGZsYXQgbGF3IGFib3V0IGEgdGhpcmQgb2YgdGhlIHRpbWUsIGFuZCBCb25mZXJyb25pLWFkanVz"
    "dGluZyB0aGVtIHB1c2hlcwogICAgdGhlIHJlcXVpcmVkIGJvb3RzdHJhcCBwZXJjZW50aWxlIG91"
    "dCB0byAwLjMlLCB3aGljaCBhIGZldyBkb3plbiBibG9ja3MgY2Fubm90CiAgICByZXNvbHZlLiBU"
    "aGUgc3VwLXN0YXRpc3RpYyBhc2tzIHRoZSBzaW11bHRhbmVvdXMgcXVlc3Rpb24gZGlyZWN0bHkg"
    "YW5kIG9ubHkgZXZlcgogICAgbmVlZHMgYSBjZW50cmFsIHBlcmNlbnRpbGUgb2YgdGhlIGJvb3Rz"
    "dHJhcCBkaXN0cmlidXRpb246CgogICAgICAgIFQgICAgICA9IG1heF9pIHxtZWFuX2kgLSBudWxs"
    "fCAvIHNlX2kKICAgICAgICBUKl9iICAgPSBtYXhfaSB8bWVhbipfaShiKSAtIG1lYW5faXwgLyBz"
    "ZV9pICAgICAgICAgIChib290c3RyYXAgd29ybGQgbnVsbCkKICAgICAgICBmbGF0ICA8PT4gIFQg"
    "PD0gcXVhbnRpbGUoVCosIGxldmVsKQoKICAgIHNlX2kgaXMgdGhlIGNsdXN0ZXItYm9vdHN0cmFw"
    "IHN0YW5kYXJkIGVycm9yIG9mIGJpbiBpLCBzbyB0aGUgc3RhdGlzdGljIGlzIG9uIGEKICAgIGNv"
    "bW1vbiBzY2FsZSBhY3Jvc3MgYmlucyBvZiB2ZXJ5IGRpZmZlcmVudCBvY2N1cGFuY3kuCiAgICAi"
    "IiIKICAgIG5fYmxvY2tzID0gbGVuKGJsb2NrcykKICAgIGlmIG5fYmxvY2tzIDwgMjoKICAgICAg"
    "ICByZXR1cm4geyJzdGF0dXMiOiAiU1VQX1RFU1RfUkVRVUlSRVNfQVRfTEVBU1RfVFdPX0JMT0NL"
    "UyIsICJwYXNzIjogRmFsc2V9CiAgICBybmcgPSBucC5yYW5kb20uZGVmYXVsdF9ybmcoaW50KHNl"
    "ZWQpKQogICAgb2JzID0gZXN0aW1hdGVzKGNvbWJpbmUoYmxvY2tzKSwgbWluX249MSkKICAgIG9i"
    "c19tZWFuID0gbnAuYXJyYXkoW3JbImNvbmRpdGlvbmFsX21lYW4iXSBmb3IgciBpbiBvYnNbImJp"
    "bnMiXV0sIG5wLmZsb2F0NjQpCiAgICBrZWVwID0gbnAuYXJyYXkoW3JbIm4iXSA+PSBtaW5fbiBm"
    "b3IgciBpbiBvYnNbImJpbnMiXV0sIGJvb2wpCgogICAgZHJhd3MgPSBucC5lbXB0eSgoaW50KG5f"
    "Ym9vdCksIE5fQklOUyksIG5wLmZsb2F0NjQpCiAgICBmb3IgYiBpbiByYW5nZShpbnQobl9ib290"
    "KSk6CiAgICAgICAgcGljayA9IHJuZy5pbnRlZ2VycygwLCBuX2Jsb2Nrcywgc2l6ZT1uX2Jsb2Nr"
    "cykKICAgICAgICBlID0gZXN0aW1hdGVzKGNvbWJpbmUoW2Jsb2Nrc1tpXSBmb3IgaSBpbiBwaWNr"
    "XSksIG1pbl9uPTEpCiAgICAgICAgZHJhd3NbYl0gPSBbclsiY29uZGl0aW9uYWxfbWVhbiJdIGZv"
    "ciByIGluIGVbImJpbnMiXV0KICAgIHNlID0gbnAubmFuc3RkKGRyYXdzLCBheGlzPTAsIGRkb2Y9"
    "MSkKICAgIHNlID0gbnAud2hlcmUoKHNlID4gMCkgJiBucC5pc2Zpbml0ZShzZSksIHNlLCBucC5u"
    "YW4pCgogICAgd2l0aCBucC5lcnJzdGF0ZShpbnZhbGlkPSJpZ25vcmUiKToKICAgICAgICB0X3N0"
    "YXIgPSBucC5uYW5tYXgobnAuYWJzKGRyYXdzWzosIGtlZXBdIC0gb2JzX21lYW5ba2VlcF0pIC8g"
    "c2Vba2VlcF0sIGF4aXM9MSkKICAgICAgICB0X29icyA9IGZsb2F0KG5wLm5hbm1heChucC5hYnMo"
    "b2JzX21lYW5ba2VlcF0gLSBmbG9hdChudWxsX3ZhbHVlKSkgLyBzZVtrZWVwXSkpCiAgICBjcml0"
    "ID0gZmxvYXQobnAubmFucGVyY2VudGlsZSh0X3N0YXIsIDEwMC4wICogbGV2ZWwpKQogICAgcGVy"
    "X2JpbiA9IFtdCiAgICBmb3IgaSBpbiByYW5nZShOX0JJTlMpOgogICAgICAgIGlmIG5vdCBrZWVw"
    "W2ldOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIHBlcl9iaW4uYXBwZW5kKHsKICAgICAg"
    "ICAgICAgImJpbiI6IEJJTl9MQUJFTFNbaV0sICJuIjogb2JzWyJiaW5zIl1baV1bIm4iXSwKICAg"
    "ICAgICAgICAgImNvbmRpdGlvbmFsX21lYW4iOiBmbG9hdChvYnNfbWVhbltpXSksCiAgICAgICAg"
    "ICAgICJzdGFuZGFyZF9lcnJvciI6IGZsb2F0KHNlW2ldKSwKICAgICAgICAgICAgInN0dWRlbnRp"
    "emVkX2RldmlhdGlvbiI6CiAgICAgICAgICAgICAgICBmbG9hdChhYnMob2JzX21lYW5baV0gLSBm"
    "bG9hdChudWxsX3ZhbHVlKSkgLyBzZVtpXSkKICAgICAgICAgICAgICAgIGlmIHNlW2ldID09IHNl"
    "W2ldIGVsc2UgZmxvYXQoIm5hbiIpfSkKICAgIHJldHVybiB7InN0YXR1cyI6ICJPSyIsICJudWxs"
    "X3ZhbHVlIjogZmxvYXQobnVsbF92YWx1ZSksCiAgICAgICAgICAgICJuX2Jsb2NrcyI6IG5fYmxv"
    "Y2tzLCAibl9ib290IjogaW50KG5fYm9vdCksICJsZXZlbCI6IGxldmVsLAogICAgICAgICAgICAi"
    "Ymluc191c2VkIjogaW50KGtlZXAuc3VtKCkpLAogICAgICAgICAgICAic3VwX3N0YXRpc3RpYyI6"
    "IHRfb2JzLCAiY3JpdGljYWxfdmFsdWUiOiBjcml0LAogICAgICAgICAgICAicGVyX2JpbiI6IHBl"
    "cl9iaW4sCiAgICAgICAgICAgICJzaW11bHRhbmVvdXNfZmxhdCI6IGJvb2wodF9vYnMgPD0gY3Jp"
    "dCksCiAgICAgICAgICAgICJwYXNzIjogYm9vbCh0X29icyA8PSBjcml0KX0K"
)
_SRC_BINNING = base64.b64decode(_SRC_BINNING_B64).decode()
open(os.path.join(SRC_DIR, "binning.py"), "w").write(_SRC_BINNING)
print('binning.py staged', len(_SRC_BINNING), 'chars')


## Module — the two frozen laws, market paths only

Frozen SBJTS target via the Base 4 pair builder, and the frozen empirical Merton iid control. Diagnostic seeds live in their own namespaces and are checked numerically against every Base 4 training and holdout seed.


In [ ]:
_SRC_LAWS_B64 = (
    "IiIiCkMtUkxTQkpUUy1DT05ETEFXLURJQUctMDEg4oCUIHRoZSB0d28gZnJvemVuIGxhd3MsIHNh"
    "bXBsZWQgYXMgTUFSS0VUIFBBVEhTIE9OTFkuCgpObyBhY3Rvciwgbm8gY3JpdGljLCBubyB0cmFp"
    "bmluZywgbm8gZXZhbHVhdGlvbiBvZiBhIHNhdmVkIHBvbGljeS4gQm90aCBzYW1wbGVycyByZXR1"
    "cm4KKFAsIE4pIGFycmF5cyBvZiB0aGUgc2FtZSBtYXJrZXQgb2JqZWN0IHRoZSBsZWFybmVyIGNv"
    "bnN1bWVzOiBgcHJvamVjdF9ldyhpbmMpYCwgdGhlCmVxdWFsbHkgd2VpZ2h0ZWQgbG9nIGluY3Jl"
    "bWVudCBhY3Jvc3MgdGhlIGZvdXIgc25hcHNob3QgYXNzZXRzLCB3aGljaCBpcyBleGFjdGx5IHRo"
    "ZQpvYmplY3QgYGNhbGlicmF0ZV9lbXBpcmljYWxfZ2JtYCB3YXMgZml0dGVkIHRvLgoiIiIKaW1w"
    "b3J0IG1hdGgKCmltcG9ydCBudW1weSBhcyBucAoKaW1wb3J0IGZyb3plbl9sb2FkZXIgYXMgRkwK"
    "CiMgU2VlZCBuYW1lc3BhY2VzIGZvciB0aGlzIGRpYWdub3N0aWMuIERlbGliZXJhdGVseSBkaXN0"
    "aW5jdCBmcm9tIGV2ZXJ5IGZyb3plbgojIG5hbWVzcGFjZSwgc28gZGlhZ25vc3RpYyBkcmF3cyBj"
    "YW4gbmV2ZXIgY29pbmNpZGUgd2l0aCBCYXNlIDQgdHJhaW5pbmcgb3IgaG9sZG91dAojIGV2YWx1"
    "YXRpb24gZHJhd3M7IHZlcmlmaWVkIG51bWVyaWNhbGx5IGJ5IGBhc3NlcnRfc2VlZF9pc29sYXRp"
    "b25gLgpOU19TQkpUUyA9ICJDT05ETEFXX0RJQUdOT1NUSUNfU0JKVFNfQkxPQ0siCk5TX01FUlRP"
    "TiA9ICJDT05ETEFXX0RJQUdOT1NUSUNfTUVSVE9OX0JMT0NLIgpGUk9aRU5fTkFNRVNQQUNFUyA9"
    "ICgiRVZBTF9FTlYiLCAiTEVBUk5FUiIsICJPUkFDTEVfRU5WIiwgIlRSQUlOX0VOViIsICJWRVJJ"
    "RllfRU5WIiwKICAgICAgICAgICAgICAgICAgICAgIkJBU0U0X0pPSU5UX1RSQUlOSU5HX1JFUExJ"
    "Q0FUSU9OIiwgIkJBU0U0X0hPTERPVVRfRU5WSVJPTk1FTlQiLAogICAgICAgICAgICAgICAgICAg"
    "ICAiQkFTRTRfRVZBTFVBVElPTl9TRUVEIikKCgpkZWYgYmxvY2tfc2VlZChuYW1lc3BhY2UsIGJs"
    "b2NrX2luZGV4KToKICAgIHJldHVybiBGTC5kZXJpdmVfc2VlZChuYW1lc3BhY2UsIGludChibG9j"
    "a19pbmRleCkpCgoKZGVmIGFzc2VydF9zZWVkX2lzb2xhdGlvbihuX2Jsb2NrcywgcHJvZmlsZT1O"
    "b25lKToKICAgICIiIk5vIGRpYWdub3N0aWMgYmxvY2sgc2VlZCBtYXkgY29pbmNpZGUgd2l0aCBh"
    "IEJhc2UgNCB0cmFpbmluZyBvciBob2xkb3V0IHNlZWQuCgogICAgVGhpcyBpcyBjaGVja2VkIGFn"
    "YWluc3QgdGhlIGFjdHVhbCBpbnRlZ2Vycywgbm90IGFyZ3VlZCBmcm9tIG5hbWVzcGFjZSBuYW1l"
    "czogYQogICAgaGFzaCBjb2xsaXNpb24gd291bGQgYmUgc2lsZW50IG90aGVyd2lzZSwgYW5kIHJl"
    "dXNpbmcgYSBob2xkb3V0IG1hcmtldCBzZWVkIHdvdWxkCiAgICBlbnRhbmdsZSBhIGRlc2NyaXB0"
    "aXZlIGRpYWdub3N0aWMgd2l0aCB0aGUgZnJvemVuIGV2YWx1YXRpb24gc2FtcGxlLgogICAgIiIi"
    "CiAgICBmcm96ZW4gPSBzZXQoKQogICAgZm9yIHJvdyBpbiBGTC5iYXNlNF90cmFpbl9wbGFuKHBy"
    "b2ZpbGUpOgogICAgICAgIGZyb3plbiB8PSB7cm93WyJsZWFybmVyX3NlZWQiXSwgcm93WyJ0cmFp"
    "bmluZ19lbnZpcm9ubWVudF9zZWVkIl0sCiAgICAgICAgICAgICAgICAgICByb3dbInRyYWluaW5n"
    "X2JhdGNoX3NlZWQiXX0KICAgIGZvciBibGsgaW4gRkwuYmFzZTRfZXZhbF9ibG9ja3MocHJvZmls"
    "ZSk6CiAgICAgICAgZnJvemVuIHw9IHtibGtbIm1hcmtldF9zZWVkIl0sIGJsa1siYWN0aW9uX3Nl"
    "ZWQiXX0KICAgIGRpYWcgPSB7YmxvY2tfc2VlZChucywgaSkKICAgICAgICAgICAgZm9yIG5zIGlu"
    "IChOU19TQkpUUywgTlNfTUVSVE9OKSBmb3IgaSBpbiByYW5nZShpbnQobl9ibG9ja3MpKX0KICAg"
    "IGNsYXNoID0gc29ydGVkKGRpYWcgJiBmcm96ZW4pCiAgICBpZiBjbGFzaDoKICAgICAgICByYWlz"
    "ZSBSdW50aW1lRXJyb3IoZiJDT05ETEFXX1NFRURfQ09MTElERVNfV0lUSF9GUk9aRU5fQkFTRTRf"
    "U0VFRFM6IHtjbGFzaH0iKQogICAgcmV0dXJuIHsibl9kaWFnbm9zdGljX3NlZWRzIjogbGVuKGRp"
    "YWcpLCAibl9mcm96ZW5fc2VlZHMiOiBsZW4oZnJvemVuKSwKICAgICAgICAgICAgImludGVyc2Vj"
    "dGlvbiI6IFtdLCAiaXNvbGF0ZWQiOiBUcnVlfQoKCmRlZiBtZXJ0b25fYmxvY2sobnMsIHJlYywg"
    "YmxvY2tfaW5kZXgsIG5fcGF0aHMsIG5fc3RlcHMpOgogICAgIiIiRnJvemVuIGVtcGlyaWNhbC1N"
    "ZXJ0b24gY29udHJvbDogaWlkIEdhdXNzaWFuIGxvZyBpbmNyZW1lbnRzLgoKICAgIENvbmRpdGlv"
    "bmFsIG1lYW4gaXMgYSBDT05TVEFOVCBieSBjb25zdHJ1Y3Rpb24sIHdoaWNoIGlzIHdoYXQgbWFr"
    "ZXMgdGhpcyB0aGUKICAgIHBvc2l0aXZlIGNvbnRyb2wgZm9yIGEgZmxhdCByZXNwb25zZSBjdXJ2"
    "ZS4KICAgICIiIgogICAgbTEgPSBmbG9hdChyZWNbIm0xX3Blcl9zdGVwX21lYW5fbG9nX2luY3Jl"
    "bWVudCJdKQogICAgc2QgPSBtYXRoLnNxcnQoZmxvYXQocmVjWyJ2MV9wZXJfc3RlcF92YXJpYW5j"
    "ZV9sb2dfaW5jcmVtZW50Il0pKQogICAgIyBOT1QgbnNbInJuZ19vZiJdOiB0aGUgZnJvemVuIGxl"
    "YXJuZXIga2VlcHMgYSBjbG9zZWQgbmFtZXNwYWNlIHJlZ2lzdHJ5IGFuZCB0aGlzCiAgICAjIGRp"
    "YWdub3N0aWMgbXVzdCBub3QgZXh0ZW5kIGl0LiBUaGUgZ2VuZXJhdG9yIGlzIHNlZWRlZCBmcm9t"
    "IHRoZSBzYW1lIGhhc2gtZGVyaXZlZAogICAgIyBTZWVkU2VxdWVuY2UgbWFjaGluZXJ5IGluc3Rl"
    "YWQsIGluIGEgbmFtZXNwYWNlIG9mIGl0cyBvd24uCiAgICBybmcgPSBucC5yYW5kb20uZGVmYXVs"
    "dF9ybmcoYmxvY2tfc2VlZChOU19NRVJUT04sIGJsb2NrX2luZGV4KSkKICAgIHJldHVybiBtMSAr"
    "IHNkICogcm5nLnN0YW5kYXJkX25vcm1hbCgoaW50KG5fcGF0aHMpLCBpbnQobl9zdGVwcykpKQoK"
    "CmRlZiBzYmp0c19ibG9jayhlbmdpbmUsIGJsb2NrX2luZGV4LCBuX3BhdGhzKToKICAgICIiIkZy"
    "b3plbiBTQkpUUyB0YXJnZXQgbGF3LCBtYXJrZXQgc2ltdWxhdGlvbiBvbmx5LgoKICAgIENhbGxz"
    "IHRoZSBmcm96ZW4gQmFzZSA0IHBhaXIgYnVpbGRlciBmb3IgdGhlIFRBUkdFVCBsYXcgYWxvbmU7"
    "IHRoZSBlbnZpcm9ubWVudAogICAgZmluZ2VycHJpbnQgbG9jayBpbnNpZGUgYHJ1bl9lbmdpbmVg"
    "IHN0aWxsIGd1YXJkcyBhZ2FpbnN0IGNhbGlicmF0aW9uIG11dGF0aW9uLgogICAgIiIiCiAgICBz"
    "ZWVkID0gYmxvY2tfc2VlZChOU19TQkpUUywgYmxvY2tfaW5kZXgpCiAgICBwYWlyID0gZW5naW5l"
    "Lm1ha2VfYmFzZTRfbWFya2V0X3BhaXIoc2VlZCwgaW50KG5fcGF0aHMpLCBsYXdzPSgiVEFSR0VU"
    "IiwpKQogICAgYmF0Y2ggPSBwYWlyW0ZMLlRBUkdFVF9MQVdfTkFNRV0KICAgIHJldHVybiBucC5h"
    "c2FycmF5KGJhdGNoLnJpc2t5X2xvZ19yZXR1cm5zLCBucC5mbG9hdDY0KSwgc2VlZAoKCmRlZiBh"
    "cjFfbXV0YXRlZF9tZXJ0b25fYmxvY2sobnMsIHJlYywgYmxvY2tfaW5kZXgsIG5fcGF0aHMsIG5f"
    "c3RlcHMsIHJobyk6CiAgICAiIiJORUdBVElWRSBDT05UUk9MIE9OTFk6IE1lcnRvbiBpbmNyZW1l"
    "bnRzIHdpdGggYW4gaW5qZWN0ZWQgQVIoMSkgZGVwZW5kZW5jZS4KCiAgICBVc2VkIHNvbGVseSB0"
    "byBzaG93IHRoZSBmbGF0bmVzcyBnYXRlIGNhbiBmYWlsLiBOZXZlciBhIGRpYWdub3N0aWMgbGF3"
    "LgogICAgIiIiCiAgICBtMSA9IGZsb2F0KHJlY1sibTFfcGVyX3N0ZXBfbWVhbl9sb2dfaW5jcmVt"
    "ZW50Il0pCiAgICBzZCA9IG1hdGguc3FydChmbG9hdChyZWNbInYxX3Blcl9zdGVwX3ZhcmlhbmNl"
    "X2xvZ19pbmNyZW1lbnQiXSkpCiAgICBybmcgPSBucC5yYW5kb20uZGVmYXVsdF9ybmcoYmxvY2tf"
    "c2VlZChOU19NRVJUT04sIDkwMDAwMCArIGludChibG9ja19pbmRleCkpKQogICAgZSA9IHJuZy5z"
    "dGFuZGFyZF9ub3JtYWwoKGludChuX3BhdGhzKSwgaW50KG5fc3RlcHMpKSkKICAgIG91dCA9IG5w"
    "LmVtcHR5X2xpa2UoZSkKICAgIG91dFs6LCAwXSA9IGVbOiwgMF0KICAgIGZvciB0IGluIHJhbmdl"
    "KDEsIGludChuX3N0ZXBzKSk6CiAgICAgICAgb3V0WzosIHRdID0gZmxvYXQocmhvKSAqIG91dFs6"
    "LCB0IC0gMV0gKyBtYXRoLnNxcnQoMS4wIC0gZmxvYXQocmhvKSAqKiAyKSAqIGVbOiwgdF0KICAg"
    "IHJldHVybiBtMSArIHNkICogb3V0Cg=="
)
_SRC_LAWS = base64.b64decode(_SRC_LAWS_B64).decode()
open(os.path.join(SRC_DIR, "laws.py"), "w").write(_SRC_LAWS)
print('laws.py staged', len(_SRC_LAWS), 'chars')


## Module — SMOKE / RESEARCH runner

Simulation, checkpointing, pooling, bootstrap and the output schema. Never constructs an actor or a critic.


In [ ]:
_SRC_RUNNER_B64 = (
    "IiIiCkMtUkxTQkpUUy1DT05ETEFXLURJQUctMDEg4oCUIFNNT0tFIC8gUkVTRUFSQ0ggcnVubmVy"
    "LgoKTWFya2V0IHNpbXVsYXRpb24gb25seS4gVGhpcyBtb2R1bGUgbmV2ZXIgY29uc3RydWN0cyBh"
    "IExpbmVhckFjdG9yLCBuZXZlciBjYWxscwphY3Rvcl9ncmFkaWVudCwgYW5kIG5ldmVyIGxvYWRz"
    "IGEgc2F2ZWQgcG9saWN5OyB0aGUgcG9saWN5IG92ZXJsYXkgaXMgYSBzZXBhcmF0ZQpyZWFkLW9u"
    "bHkgbW9kdWxlIHRoYXQgY29uc3VtZXMgYWxyZWFkeS1hY2NlcHRlZCBDU1YgZXZpZGVuY2UuCgpS"
    "ZXN1bWFiaWxpdHkgaXMgcGVyIChsYXcsIGJsb2NrKTogZWFjaCBibG9jayB3cml0ZXMgaXRzIHN1"
    "ZmZpY2llbnQgc3RhdGlzdGljcyB0byBpdHMgb3duCmNoZWNrcG9pbnQgZmlsZSwgYW5kIGEgcmVz"
    "dW1lZCBydW4gcmVjb21iaW5lcyBjaGVja3BvaW50cyBpbnN0ZWFkIG9mIHJlLXNpbXVsYXRpbmcu"
    "CkJlY2F1c2UgdGhlIHBvb2xlZCBlc3RpbWF0ZSBpcyBhbiBleGFjdCBzdW0gb2YgYmxvY2sgc3Rh"
    "dGlzdGljcywgYSByZXN1bWVkIHJ1biBhbmQgYW4KdW5pbnRlcnJ1cHRlZCBydW4gYWdyZWUgYml0"
    "IGZvciBiaXQuCiIiIgppbXBvcnQgY3N2CmltcG9ydCBqc29uCmltcG9ydCBvcwppbXBvcnQgdGlt"
    "ZQoKaW1wb3J0IG51bXB5IGFzIG5wCgppbXBvcnQgY29tcGFyYXRvcl9jb25maWcgYXMgQ0MKaW1w"
    "b3J0IGZyb3plbl9sb2FkZXIgYXMgRkwKaW1wb3J0IGJpbm5pbmcgYXMgQgppbXBvcnQgbGF3cyBh"
    "cyBMCgpMQVdfU0JKVFMgPSAiU0JKVFNfVEFSR0VUIgpMQVdfTUVSVE9OID0gIk1FUlRPTl9FTVBJ"
    "UklDQUxfR0JNIgpMQVdTID0gKExBV19TQkpUUywgTEFXX01FUlRPTikKClBST0ZJTEVTID0gewog"
    "ICAgIlNNT0tFIjogeyJuX2Jsb2NrcyI6IDYsICJuX3BhdGhzIjogNjQsICJuX2Jvb3QiOiAyMDAw"
    "LAogICAgICAgICAgICAgICJub3RlIjogInRpbnkgZnJvemVuLWVuZ2luZSBmaXh0dXJlLCBDUFUs"
    "IGV4ZXJjaXNlcyBldmVyeSBiaW4gYW5kIHRoZSAiCiAgICAgICAgICAgICAgICAgICAgICAid2hv"
    "bGUgb3V0cHV0IHNjaGVtYSJ9LAogICAgIyBQcm9wb3NlZCBib3VuZGVkIFJFU0VBUkNIIGJ1ZGdl"
    "dC4gQmxvY2tzIGFyZSBmYXZvdXJlZCBvdmVyIHBhdGhzLXBlci1ibG9jazoKICAgICMgdW5jZXJ0"
    "YWludHkgaXMgcXVhbnRpZmllZCBhdCB0aGUgQkxPQ0sgbGV2ZWwsIHNvIDY0IHJlc2FtcGxpbmcg"
    "dW5pdHMgYXQgMzA3MgogICAgIyBwYXRocyBlYWNoIGlzIGEgYmV0dGVyIGRlc2lnbiB0aGFuIDMy"
    "IHVuaXRzIGF0IDYxNDQgZm9yIHRoZSBzYW1lIHRvdGFsIGNvc3QuCiAgICAiUkVTRUFSQ0giOiB7"
    "Im5fYmxvY2tzIjogNjQsICJuX3BhdGhzIjogMzA3MiwgIm5fYm9vdCI6IDQwMDAsCiAgICAgICAg"
    "ICAgICAgICAgIm5vdGUiOiAicHJvcG9zZWQgYm91bmRlZCBidWRnZXQ7IHJlcXVpcmVzIFBNTyBh"
    "dXRob3JpemF0aW9uIGFuZCBpcyAiCiAgICAgICAgICAgICAgICAgICAgICAgICAiZXhlY3V0ZWQg"
    "YnkgdGhlIHVzZXIgb24gQ29sYWIsIG5ldmVyIGJ5IENsYXVkZSJ9LAp9CgoKZGVmIF9ja3B0X3Bh"
    "dGgob3V0X2RpciwgbGF3LCBibG9jayk6CiAgICByZXR1cm4gb3MucGF0aC5qb2luKG91dF9kaXIs"
    "ICJibG9ja3MiLCBmIntsYXd9X19ibG9ja3tibG9jazowNGR9Lmpzb24iKQoKCmRlZiBfc2F2ZV9i"
    "bG9jayhwYXRoLCBsYXcsIGJsb2NrLCBzZWVkLCBzdGF0cywgZWxhcHNlZF9zLCBuX3BhdGhzLCBu"
    "X3N0ZXBzKToKICAgIG9zLm1ha2VkaXJzKG9zLnBhdGguZGlybmFtZShwYXRoKSwgZXhpc3Rfb2s9"
    "VHJ1ZSkKICAgIHBheWxvYWQgPSB7CiAgICAgICAgImxhdyI6IGxhdywgImJsb2NrX2luZGV4Ijog"
    "aW50KGJsb2NrKSwgInNlZWQiOiBpbnQoc2VlZCksCiAgICAgICAgIm5fcGF0aHMiOiBpbnQobl9w"
    "YXRocyksICJuX3N0ZXBzIjogaW50KG5fc3RlcHMpLAogICAgICAgICJlbGFwc2VkX3MiOiByb3Vu"
    "ZChmbG9hdChlbGFwc2VkX3MpLCAzKSwKICAgICAgICAiYmlucyI6IHN0YXRzWyJiaW5zIl0udG9s"
    "aXN0KCksICJnbG9iYWwiOiBzdGF0c1siZ2xvYmFsIl0udG9saXN0KCksCiAgICAgICAgImJpbl9m"
    "aWVsZHMiOiBsaXN0KEIuQklOX0ZJRUxEUyksICJnbG9iYWxfZmllbGRzIjogbGlzdChCLkdMT0JB"
    "TF9GSUVMRFMpLAogICAgICAgICJiaW5fbGFiZWxzIjogbGlzdChCLkJJTl9MQUJFTFMpLAogICAg"
    "fQogICAgdG1wID0gcGF0aCArICIudG1wIgogICAgd2l0aCBvcGVuKHRtcCwgInciKSBhcyBmaDoK"
    "ICAgICAgICBqc29uLmR1bXAocGF5bG9hZCwgZmgpCiAgICBvcy5yZXBsYWNlKHRtcCwgcGF0aCkg"
    "ICAgICAgICAgIyBhdG9taWMsIHNvIGFuIGludGVycnVwdGVkIHdyaXRlIGNhbm5vdCBiZSByZXN1"
    "bWVkCiAgICByZXR1cm4gcGF5bG9hZAoKCmRlZiBfbG9hZF9ibG9jayhwYXRoKToKICAgIHdpdGgg"
    "b3BlbihwYXRoKSBhcyBmaDoKICAgICAgICBkID0ganNvbi5sb2FkKGZoKQogICAgaWYgbGlzdChk"
    "LmdldCgiYmluX2xhYmVscyIsIFtdKSkgIT0gbGlzdChCLkJJTl9MQUJFTFMpOgogICAgICAgIHJh"
    "aXNlIFJ1bnRpbWVFcnJvcihmIkNIRUNLUE9JTlRfQklOX1NDSEVNRV9NSVNNQVRDSDoge3BhdGh9"
    "IikKICAgIHJldHVybiBkLCB7ImJpbnMiOiBucC5hc2FycmF5KGRbImJpbnMiXSwgbnAuZmxvYXQ2"
    "NCksCiAgICAgICAgICAgICAgICJnbG9iYWwiOiBucC5hc2FycmF5KGRbImdsb2JhbCJdLCBucC5m"
    "bG9hdDY0KX0KCgpkZWYgc2ltdWxhdGVfYmxvY2tzKG5zLCBjYWwsIHJlYywgb3V0X2RpciwgcnVu"
    "X21vZGUsIG5fYmxvY2tzLCBuX3BhdGhzLAogICAgICAgICAgICAgICAgICAgIGVuZ2luZT1Ob25l"
    "LCByZXN1bWU9VHJ1ZSwgcHJvZ3Jlc3M9VHJ1ZSk6CiAgICAiIiJTaW11bGF0ZSAob3IgcmVzdW1l"
    "KSBldmVyeSBibG9jayBmb3IgYm90aCBsYXdzIGFuZCByZXR1cm4gdGhlaXIgc3RhdGlzdGljcy4i"
    "IiIKICAgIG0xID0gZmxvYXQocmVjWyJtMV9wZXJfc3RlcF9tZWFuX2xvZ19pbmNyZW1lbnQiXSkK"
    "ICAgIHYxID0gZmxvYXQocmVjWyJ2MV9wZXJfc3RlcF92YXJpYW5jZV9sb2dfaW5jcmVtZW50Il0p"
    "CiAgICBxX3RhaWwgPSBCLm1lcnRvbl90YWlsX3F1YW50aWxlKG0xLCB2MSwgMC4wNSkKICAgIG5f"
    "c3RlcHMgPSBpbnQobnNbIk5fU1RFUFMiXSkKICAgIGVuZ2luZSA9IGVuZ2luZSBvciBGTC5CYXNl"
    "NEVuZ2luZShucywgY2FsKQogICAgTC5hc3NlcnRfc2VlZF9pc29sYXRpb24obWF4KGludChuX2Js"
    "b2NrcyksIDEpKQoKICAgIGJsb2NrcyA9IHtsYXc6IFtdIGZvciBsYXcgaW4gTEFXU30KICAgIGF0"
    "dGVtcHRzID0gW10KICAgIGZvciBsYXcgaW4gTEFXUzoKICAgICAgICBmb3IgYiBpbiByYW5nZShp"
    "bnQobl9ibG9ja3MpKToKICAgICAgICAgICAgcGF0aCA9IF9ja3B0X3BhdGgob3V0X2RpciwgbGF3"
    "LCBiKQogICAgICAgICAgICBpZiByZXN1bWUgYW5kIG9zLnBhdGguZXhpc3RzKHBhdGgpOgogICAg"
    "ICAgICAgICAgICAgbWV0YSwgc3RhdHMgPSBfbG9hZF9ibG9jayhwYXRoKQogICAgICAgICAgICAg"
    "ICAgbWV0YVsicmVzdW1lZCJdID0gVHJ1ZQogICAgICAgICAgICBlbHNlOgogICAgICAgICAgICAg"
    "ICAgdDAgPSB0aW1lLnRpbWUoKQogICAgICAgICAgICAgICAgaWYgbGF3ID09IExBV19TQkpUUzoK"
    "ICAgICAgICAgICAgICAgICAgICByLCBzZWVkID0gTC5zYmp0c19ibG9jayhlbmdpbmUsIGIsIG5f"
    "cGF0aHMpCiAgICAgICAgICAgICAgICBlbHNlOgogICAgICAgICAgICAgICAgICAgIHIgPSBMLm1l"
    "cnRvbl9ibG9jayhucywgcmVjLCBiLCBuX3BhdGhzLCBuX3N0ZXBzKQogICAgICAgICAgICAgICAg"
    "ICAgIHNlZWQgPSBMLmJsb2NrX3NlZWQoTC5OU19NRVJUT04sIGIpCiAgICAgICAgICAgICAgICBz"
    "dGF0cyA9IEIuYmxvY2tfc3RhdGlzdGljcyhyLCBtMSwgdjEsIHFfdGFpbCkKICAgICAgICAgICAg"
    "ICAgIG1ldGEgPSBfc2F2ZV9ibG9jayhwYXRoLCBsYXcsIGIsIHNlZWQsIHN0YXRzLCB0aW1lLnRp"
    "bWUoKSAtIHQwLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIG5fcGF0aHMsIHIu"
    "c2hhcGVbMV0pCiAgICAgICAgICAgICAgICBtZXRhWyJyZXN1bWVkIl0gPSBGYWxzZQogICAgICAg"
    "ICAgICAgICAgaWYgcHJvZ3Jlc3M6CiAgICAgICAgICAgICAgICAgICAgcHJpbnQoZiIgIFt7cnVu"
    "X21vZGV9XSB7bGF3fSBibG9jayB7YiArIDF9L3tuX2Jsb2Nrc30gIgogICAgICAgICAgICAgICAg"
    "ICAgICAgICAgIGYie21ldGFbJ2VsYXBzZWRfcyddOi4xZn1zIiwgZmx1c2g9VHJ1ZSkKICAgICAg"
    "ICAgICAgYmxvY2tzW2xhd10uYXBwZW5kKHN0YXRzKQogICAgICAgICAgICBhdHRlbXB0cy5hcHBl"
    "bmQoe2s6IG1ldGEuZ2V0KGspIGZvciBrIGluCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAg"
    "KCJsYXciLCAiYmxvY2tfaW5kZXgiLCAic2VlZCIsICJuX3BhdGhzIiwgIm5fc3RlcHMiLAogICAg"
    "ICAgICAgICAgICAgICAgICAgICAgICAgICAiZWxhcHNlZF9zIiwgInJlc3VtZWQiKX0pCiAgICBy"
    "ZXR1cm4gYmxvY2tzLCBhdHRlbXB0cywgeyJxX3RhaWxfbWVydG9uXzVwY3QiOiBxX3RhaWwsICJt"
    "MSI6IG0xLCAidjEiOiB2MX0KCgpkZWYgYW5hbHlzZShibG9ja3MsIG5fYm9vdCwgc2VlZD0yMDI2"
    "MDkyMik6CiAgICBvdXQgPSB7fQogICAgZm9yIGxhdywgYmxrcyBpbiBibG9ja3MuaXRlbXMoKToK"
    "ICAgICAgICBlc3QgPSBCLmVzdGltYXRlcyhCLmNvbWJpbmUoYmxrcykpCiAgICAgICAgYm9vdCA9"
    "IEIuY2x1c3Rlcl9ib290c3RyYXAoYmxrcywgbl9ib290PW5fYm9vdCwgc2VlZD1zZWVkKQogICAg"
    "ICAgIG91dFtsYXddID0geyJlc3RpbWF0ZXMiOiBCLmF0dGFjaF9pbnRlcnZhbHMoZXN0LCBib290"
    "KSwKICAgICAgICAgICAgICAgICAgICAiYm9vdHN0cmFwIjoge2s6IHYgZm9yIGssIHYgaW4gYm9v"
    "dC5pdGVtcygpCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBpZiBrIG5vdCBpbiAo"
    "ImxvIiwgImhpIil9fQogICAgcmV0dXJuIG91dAoKCmRlZiBfd3JpdGVfY3N2KHBhdGgsIHJvd3Ms"
    "IGZpZWxkcywgcnVuX21vZGUsIGV2aWRlbmNlX3Jvb3QpOgogICAgQ0MuYXNzZXJ0X25hbWVzcGFj"
    "ZV9pc29sYXRpb24ocGF0aCwgcnVuX21vZGUsIGV2aWRlbmNlX3Jvb3QpCiAgICBvcy5tYWtlZGly"
    "cyhvcy5wYXRoLmRpcm5hbWUocGF0aCksIGV4aXN0X29rPVRydWUpCiAgICB3aXRoIG9wZW4ocGF0"
    "aCwgInciLCBuZXdsaW5lPSIiKSBhcyBmaDoKICAgICAgICB3ID0gY3N2LkRpY3RXcml0ZXIoZmgs"
    "IGZpZWxkbmFtZXM9ZmllbGRzKQogICAgICAgIHcud3JpdGVoZWFkZXIoKQogICAgICAgIGZvciBy"
    "IGluIHJvd3M6CiAgICAgICAgICAgIHcud3JpdGVyb3coe2s6IHIuZ2V0KGspIGZvciBrIGluIGZp"
    "ZWxkc30pCiAgICByZXR1cm4gcGF0aAoKCmRlZiB3cml0ZV9vdXRwdXRzKG91dF9kaXIsIGFuYWx5"
    "c2lzLCBhdHRlbXB0cywgY29uc3RzLCBydW5fbW9kZSwgZXZpZGVuY2Vfcm9vdCwKICAgICAgICAg"
    "ICAgICAgICAgZmluZ2VycHJpbnQsIHByb2ZpbGVfdXNlZCk6CiAgICAiIiJFbWl0IHRoZSBleGFj"
    "dCBzY2hlbWEgdGhlIHRpY2tldCdzIHNlY3Rpb24gOCBuYW1lcy4iIiIKICAgIHdyaXR0ZW4gPSBb"
    "XQoKICAgIGRlZiByb3dzX2ZvcihrZXlzLCBleHRyYT0oKSk6CiAgICAgICAgcm93cyA9IFtdCiAg"
    "ICAgICAgZm9yIGxhdyBpbiBMQVdTOgogICAgICAgICAgICBmb3IgciBpbiBhbmFseXNpc1tsYXdd"
    "WyJlc3RpbWF0ZXMiXVsiYmlucyJdOgogICAgICAgICAgICAgICAgcm93ID0geyJsYXciOiBsYXcs"
    "ICJiaW4iOiByWyJiaW4iXSwgImJpbl9pbmRleCI6IHJbImJpbl9pbmRleCJdLAogICAgICAgICAg"
    "ICAgICAgICAgICAgICJuIjogclsibiJdLCAibWVhbl9sYWdnZWRfcmV0dXJuIjogclsibWVhbl9s"
    "YWdnZWRfcmV0dXJuIl19CiAgICAgICAgICAgICAgICBmb3IgayBpbiBrZXlzOgogICAgICAgICAg"
    "ICAgICAgICAgIHJvd1trXSA9IHIuZ2V0KGspCiAgICAgICAgICAgICAgICAgICAgcm93W2sgKyAi"
    "X2xvIl0gPSByLmdldChrICsgIl9sbyIpCiAgICAgICAgICAgICAgICAgICAgcm93W2sgKyAiX2hp"
    "Il0gPSByLmdldChrICsgIl9oaSIpCiAgICAgICAgICAgICAgICBmb3IgayBpbiBleHRyYToKICAg"
    "ICAgICAgICAgICAgICAgICByb3dba10gPSByLmdldChrKQogICAgICAgICAgICAgICAgcm93cy5h"
    "cHBlbmQocm93KQogICAgICAgIHJldHVybiByb3dzCgogICAgYmFzZSA9IFsibGF3IiwgImJpbiIs"
    "ICJiaW5faW5kZXgiLCAibiIsICJtZWFuX2xhZ2dlZF9yZXR1cm4iXQogICAgbWVhbl9rZXlzID0g"
    "WyJjb25kaXRpb25hbF9tZWFuIiwgImRldmlhdGlvbl9mcm9tX3VuY29uZGl0aW9uYWxfbWVhbiJd"
    "CiAgICB3cml0dGVuLmFwcGVuZChfd3JpdGVfY3N2KAogICAgICAgIG9zLnBhdGguam9pbihvdXRf"
    "ZGlyLCAiY29uZGl0aW9uYWxfbWVhbl9iaW5zLmNzdiIpLAogICAgICAgIHJvd3NfZm9yKG1lYW5f"
    "a2V5cyksIGJhc2UgKyBbayArIHMgZm9yIGsgaW4gbWVhbl9rZXlzCiAgICAgICAgICAgICAgICAg"
    "ICAgICAgICAgICAgICAgICAgICBmb3IgcyBpbiAoIiIsICJfbG8iLCAiX2hpIildLAogICAgICAg"
    "IHJ1bl9tb2RlLCBldmlkZW5jZV9yb290KSkKICAgIHdyaXR0ZW4uYXBwZW5kKF93cml0ZV9jc3Yo"
    "CiAgICAgICAgb3MucGF0aC5qb2luKG91dF9kaXIsICJjb25kaXRpb25hbF92YXJpYW5jZV9iaW5z"
    "LmNzdiIpLAogICAgICAgIHJvd3NfZm9yKFsiY29uZGl0aW9uYWxfdmFyaWFuY2UiXSksCiAgICAg"
    "ICAgYmFzZSArIFsiY29uZGl0aW9uYWxfdmFyaWFuY2UiLCAiY29uZGl0aW9uYWxfdmFyaWFuY2Vf"
    "bG8iLAogICAgICAgICAgICAgICAgImNvbmRpdGlvbmFsX3ZhcmlhbmNlX2hpIl0sIHJ1bl9tb2Rl"
    "LCBldmlkZW5jZV9yb290KSkKICAgIHdyaXR0ZW4uYXBwZW5kKF93cml0ZV9jc3YoCiAgICAgICAg"
    "b3MucGF0aC5qb2luKG91dF9kaXIsICJ0YWlsX3Byb2JhYmlsaXR5X2JpbnMuY3N2IiksCiAgICAg"
    "ICAgcm93c19mb3IoWyJ0YWlsX3Byb2JhYmlsaXR5Il0pLAogICAgICAgIGJhc2UgKyBbInRhaWxf"
    "cHJvYmFiaWxpdHkiLCAidGFpbF9wcm9iYWJpbGl0eV9sbyIsICJ0YWlsX3Byb2JhYmlsaXR5X2hp"
    "Il0sCiAgICAgICAgcnVuX21vZGUsIGV2aWRlbmNlX3Jvb3QpKQogICAgd3JpdHRlbi5hcHBlbmQo"
    "X3dyaXRlX2NzdigKICAgICAgICBvcy5wYXRoLmpvaW4ob3V0X2RpciwgInNpbXVsYXRpb25fYXR0"
    "ZW1wdHMuY3N2IiksIGF0dGVtcHRzLAogICAgICAgIFsibGF3IiwgImJsb2NrX2luZGV4IiwgInNl"
    "ZWQiLCAibl9wYXRocyIsICJuX3N0ZXBzIiwgImVsYXBzZWRfcyIsICJyZXN1bWVkIl0sCiAgICAg"
    "ICAgcnVuX21vZGUsIGV2aWRlbmNlX3Jvb3QpKQoKICAgIHN1bW1hcnkgPSB7CiAgICAgICAgImRp"
    "YWdub3N0aWNfaWQiOiAiQy1STFNCSlRTLUNPTkRMQVctRElBRy0wMSIsCiAgICAgICAgInJ1bl9t"
    "b2RlIjogcnVuX21vZGUsCiAgICAgICAgInByb2ZpbGUiOiBwcm9maWxlX3VzZWQsCiAgICAgICAg"
    "ImlzX3NjaWVudGlmaWNfZXZpZGVuY2UiOiBydW5fbW9kZSA9PSAiUkVTRUFSQ0giLAogICAgICAg"
    "ICJpc19yZXNlYXJjaF9zY2FsZV9leGVjdXRpb24iOiBydW5fbW9kZSA9PSAiUkVTRUFSQ0giLAog"
    "ICAgICAgICJjbGFpbV9zdGF0dXMiOiAiRVhQTE9SQVRPUllfTUVDSEFOSVNNX09OTFkiLAogICAg"
    "ICAgICJub19wb2xpY3lfdHJhaW5pbmdfb3JfZXZhbHVhdGlvbiI6IFRydWUsCiAgICAgICAgImJp"
    "bm5pbmciOiB7InpfZWRnZXMiOiBbc3RyKHgpIGZvciB4IGluIEIuWl9FREdFU10sCiAgICAgICAg"
    "ICAgICAgICAgICAgImxhYmVscyI6IGxpc3QoQi5CSU5fTEFCRUxTKSwKICAgICAgICAgICAgICAg"
    "ICAgICAic3RhbmRhcmRpc2F0aW9uIjogInogPSAocl9wcmV2IC0gbTEpIC8gc3FydCh2MSkgd2l0"
    "aCB0aGUgRlJPWkVOICIKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgImVt"
    "cGlyaWNhbCBNZXJ0b24gb25lLXN0ZXAgY2FsaWJyYXRpb24sIGFwcGxpZWQgIgogICAgICAgICAg"
    "ICAgICAgICAgICAgICAgICAgICAgICAgICAgICAiaWRlbnRpY2FsbHkgdG8gYm90aCBsYXdzIiwK"
    "ICAgICAgICAgICAgICAgICAgICAiZWRnZV9jb252ZW50aW9uIjogInJpZ2h0LWNsb3NlZDogYSB2"
    "YWx1ZSBvbiBhbiBlZGdlIGZhbGxzIGluIHRoZSAiCiAgICAgICAgICAgICAgICAgICAgICAgICAg"
    "ICAgICAgICAgICAgICJsb3dlciBiaW4ifSwKICAgICAgICAiY29uc3RhbnRzIjogY29uc3RzLAog"
    "ICAgICAgICJsYXdzIjoge30sCiAgICB9CiAgICBmb3IgbGF3IGluIExBV1M6CiAgICAgICAgZSA9"
    "IGFuYWx5c2lzW2xhd11bImVzdGltYXRlcyJdCiAgICAgICAgc3VtbWFyeVsibGF3cyJdW2xhd10g"
    "PSB7CiAgICAgICAgICAgIGs6IGUuZ2V0KGspIGZvciBrIGluCiAgICAgICAgICAgICgidW5jb25k"
    "aXRpb25hbF9tZWFuIiwgInVuY29uZGl0aW9uYWxfbWVhbl9sbyIsICJ1bmNvbmRpdGlvbmFsX21l"
    "YW5faGkiLAogICAgICAgICAgICAgInVuY29uZGl0aW9uYWxfdmFyaWFuY2UiLCAidW5jb25kaXRp"
    "b25hbF92YXJpYW5jZV9sbyIsCiAgICAgICAgICAgICAidW5jb25kaXRpb25hbF92YXJpYW5jZV9o"
    "aSIsICJsYWdfc2xvcGUiLCAibGFnX3Nsb3BlX2xvIiwgImxhZ19zbG9wZV9oaSIsCiAgICAgICAg"
    "ICAgICAibGFnMV9hdXRvY292YXJpYW5jZSIsICJsYWcxX2F1dG9jb3JyZWxhdGlvbiIsCiAgICAg"
    "ICAgICAgICAibGFnMV9hdXRvY29ycmVsYXRpb25fbG8iLCAibGFnMV9hdXRvY29ycmVsYXRpb25f"
    "aGkiLAogICAgICAgICAgICAgIm5fcmV0dXJucyIsICJuX3BhaXJzIil9CiAgICAgICAgc3VtbWFy"
    "eVsibGF3cyJdW2xhd11bImJvb3RzdHJhcCJdID0gYW5hbHlzaXNbbGF3XVsiYm9vdHN0cmFwIl0K"
    "ICAgIHN1bW1hcnlbInNvdXJjZV9maW5nZXJwcmludCJdID0gZmluZ2VycHJpbnQKICAgIHAgPSBv"
    "cy5wYXRoLmpvaW4ob3V0X2RpciwgImxhd19zdW1tYXJ5Lmpzb24iKQogICAgQ0MuYXNzZXJ0X25h"
    "bWVzcGFjZV9pc29sYXRpb24ocCwgcnVuX21vZGUsIGV2aWRlbmNlX3Jvb3QpCiAgICBDQy53cml0"
    "ZV9qc29uKHAsIHN1bW1hcnksIHJ1bl9tb2RlLCAibGF3X3N1bW1hcnkiLCBldmlkZW5jZV9yb290"
    "KQogICAgd3JpdHRlbi5hcHBlbmQocCkKCiAgICBwID0gb3MucGF0aC5qb2luKG91dF9kaXIsICJo"
    "YXJkd2FyZV9tYW5pZmVzdC5qc29uIikKICAgIENDLndyaXRlX2pzb24ocCwgQ0MuaGFyZHdhcmVf"
    "bWFuaWZlc3QoKSwgcnVuX21vZGUsICJoYXJkd2FyZSIsIGV2aWRlbmNlX3Jvb3QpCiAgICB3cml0"
    "dGVuLmFwcGVuZChwKQogICAgcCA9IG9zLnBhdGguam9pbihvdXRfZGlyLCAic291cmNlX2Zpbmdl"
    "cnByaW50Lmpzb24iKQogICAgQ0Mud3JpdGVfanNvbihwLCBmaW5nZXJwcmludCwgcnVuX21vZGUs"
    "ICJmaW5nZXJwcmludCIsIGV2aWRlbmNlX3Jvb3QpCiAgICB3cml0dGVuLmFwcGVuZChwKQogICAg"
    "cmV0dXJuIHdyaXR0ZW4sIHN1bW1hcnkK"
)
_SRC_RUNNER = base64.b64decode(_SRC_RUNNER_B64).decode()
open(os.path.join(SRC_DIR, "runner.py"), "w").write(_SRC_RUNNER)
print('runner.py staged', len(_SRC_RUNNER), 'chars')


## Module — read-only policy overlay

Joins the conditional-mean curve to the already-accepted policy response tables on the lagged-return axis. Direction only.


In [ ]:
_SRC_OVERLAY_B64 = (
    "IiIiCkMtUkxTQkpUUy1DT05ETEFXLURJQUctMDEg4oCUIHBvbGljeSBvdmVybGF5LCBSRUFEIE9O"
    "TFkuCgpDb25zdW1lcyBhbHJlYWR5LWFjY2VwdGVkIGNvbXBhcmF0b3IgZXZpZGVuY2UuIE5vIHBv"
    "bGljeSBpcyBsb2FkZWQsIHJldHJhaW5lZCBvcgpyZS1ldmFsdWF0ZWQgaGVyZTsgdGhpcyBtb2R1"
    "bGUgb25seSBqb2lucyB0d28gZXhpc3RpbmcgdGFibGVzIG9uIHRoZSBsYWdnZWQtcmV0dXJuIGF4"
    "aXMuCiIiIgppbXBvcnQgY3N2CmltcG9ydCBtYXRoCmltcG9ydCBvcwoKaW1wb3J0IGJpbm5pbmcg"
    "YXMgQgoKUkVQTyA9ICIvaG9tZS91c2VyL1BIRC1USEVTSVMvcmxfc2JqdHMiClNVUkZBQ0UgPSBv"
    "cy5wYXRoLmpvaW4oUkVQTywgImV2aWRlbmNlL21lcnRvbl9jb21wYXJhdG9yX3YxL3Jlc2VhcmNo"
    "IiwKICAgICAgICAgICAgICAgICAgICAgICAicG9saWN5X3Jlc3BvbnNlX3N1cmZhY2UuY3N2IikK"
    "U0xPUEVTID0gb3MucGF0aC5qb2luKFJFUE8sICJldmlkZW5jZS9tZXJ0b25fY29tcGFyYXRvcl92"
    "MS9yZXNlYXJjaCIsCiAgICAgICAgICAgICAgICAgICAgICAicG9saWN5X3Jlc3BvbnNlX3Nsb3Bl"
    "cy5jc3YiKQoKCmRlZiBfcmVhZChwYXRoKToKICAgIHdpdGggb3BlbihwYXRoKSBhcyBmaDoKICAg"
    "ICAgICByZXR1cm4gbGlzdChjc3YuRGljdFJlYWRlcihmaCkpCgoKZGVmIGJ1aWxkX292ZXJsYXko"
    "YW5hbHlzaXMsIG0xLCB2MSwgbGF3X2Zvcl9jdXJ2ZT0iU0JKVFNfVEFSR0VUIik6CiAgICAiIiJK"
    "dXh0YXBvc2UgdGhlIGZyb3plbiBjb25kaXRpb25hbC1tZWFuIGN1cnZlIHdpdGggdGhlIGFjY2Vw"
    "dGVkIGFjdGlvbiByZXNwb25zZS4KCiAgICBUaGUgam9pbiBpcyBvbiB0aGUgbGFnZ2VkIHJldHVy"
    "bjogdGhlIHBvbGljeSBzdXJmYWNlIGlzIHRhYnVsYXRlZCBvbiBhIHJhdwogICAgbGFnZ2VkLXJl"
    "dHVybiBncmlkLCB3aGljaCBpcyBtYXBwZWQgaW50byB0aGUgZGlhZ25vc3RpYydzIHogYmlucyB1"
    "c2luZyB0aGUgc2FtZQogICAgZnJvemVuIE1lcnRvbiBjYWxpYnJhdGlvbiB1c2VkIGV2ZXJ5d2hl"
    "cmUgZWxzZSBpbiB0aGlzIHBhY2thZ2UuCiAgICAiIiIKICAgIHNkID0gbWF0aC5zcXJ0KGZsb2F0"
    "KHYxKSkKICAgIGN1cnZlID0ge3JbImJpbl9pbmRleCJdOiByIGZvciByIGluIGFuYWx5c2lzW2xh"
    "d19mb3JfY3VydmVdWyJlc3RpbWF0ZXMiXVsiYmlucyJdfQogICAgcm93cyA9IFtdCiAgICBmb3Ig"
    "ciBpbiBfcmVhZChTVVJGQUNFKToKICAgICAgICBpZiByWyJheGlzIl0gIT0gImxhZ19yZXR1cm4i"
    "OgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIHggPSBmbG9hdChyWyJ4Il0pCiAgICAgICAg"
    "eiA9ICh4IC0gZmxvYXQobTEpKSAvIHNkCiAgICAgICAgYmkgPSBpbnQoQi5iaW5faW5kZXgoW3pd"
    "KVswXSkKICAgICAgICBjID0gY3VydmUuZ2V0KGJpLCB7fSkKICAgICAgICByb3dzLmFwcGVuZCh7"
    "CiAgICAgICAgICAgICJjb25zdHJhaW50IjogclsiY29uc3RyYWludCJdLAogICAgICAgICAgICAi"
    "YXJtIjogclsiYXJtIl0sCiAgICAgICAgICAgICJsYWdfcmV0dXJuX3giOiB4LAogICAgICAgICAg"
    "ICAiel9vZl94IjogeiwKICAgICAgICAgICAgImJpbiI6IEIuQklOX0xBQkVMU1tiaV0sCiAgICAg"
    "ICAgICAgICJiaW5faW5kZXgiOiBiaSwKICAgICAgICAgICAgImxhd19jb25kaXRpb25hbF9tZWFu"
    "X2luX2JpbiI6IGMuZ2V0KCJjb25kaXRpb25hbF9tZWFuIiksCiAgICAgICAgICAgICJsYXdfY29u"
    "ZGl0aW9uYWxfbWVhbl9sbyI6IGMuZ2V0KCJjb25kaXRpb25hbF9tZWFuX2xvIiksCiAgICAgICAg"
    "ICAgICJsYXdfY29uZGl0aW9uYWxfbWVhbl9oaSI6IGMuZ2V0KCJjb25kaXRpb25hbF9tZWFuX2hp"
    "IiksCiAgICAgICAgICAgICJsYXdfYmluX24iOiBjLmdldCgibiIpLAogICAgICAgICAgICAicG9s"
    "aWN5X21lYW5fYWN0aW9uIjogZmxvYXQoclsibWVhbl9hY3Rpb24iXSksCiAgICAgICAgICAgICJw"
    "b2xpY3lfbWVhbl9hY3Rpb25fc2QiOiBmbG9hdChyWyJzZCJdKSwKICAgICAgICB9KQogICAgcm93"
    "cy5zb3J0KGtleT1sYW1iZGEgZDogKGRbImNvbnN0cmFpbnQiXSwgZFsiYXJtIl0sIGRbImxhZ19y"
    "ZXR1cm5feCJdKSkKCiAgICBzbG9wZXMgPSBbXQogICAgY3VydmVfc2xvcGUgPSBhbmFseXNpc1ts"
    "YXdfZm9yX2N1cnZlXVsiZXN0aW1hdGVzIl0uZ2V0KCJsYWdfc2xvcGUiKQogICAgZm9yIHIgaW4g"
    "X3JlYWQoU0xPUEVTKToKICAgICAgICBpZiByWyJheGlzIl0gIT0gImxhZ19yZXR1cm4iOgogICAg"
    "ICAgICAgICBjb250aW51ZQogICAgICAgIHBzID0gZmxvYXQoclsibWVhbl9zbG9wZSJdKQogICAg"
    "ICAgIGNvbnNpc3RlbnQgPSBOb25lCiAgICAgICAgaWYgY3VydmVfc2xvcGUgaXMgbm90IE5vbmUg"
    "YW5kIGN1cnZlX3Nsb3BlID09IGN1cnZlX3Nsb3BlIGFuZCBwcyA9PSBwczoKICAgICAgICAgICAg"
    "IyAiY29uc2lzdGVudCIgaGVyZSBtZWFucyBvbmx5OiB0aGUgbGVhcm5lZCBhY3Rpb24gbW92ZXMg"
    "aW4gdGhlIGRpcmVjdGlvbgogICAgICAgICAgICAjIHRoYXQgdGhlIGxhdydzIG93biBjb25kaXRp"
    "b25hbCBtZWFuIHdvdWxkIHJld2FyZCBhdCBmaXJzdCBvcmRlciwgc2luY2UKICAgICAgICAgICAg"
    "IyBleHBlY3RlZCBncm93dGggaW5jcmVhc2VzIGluIHRoZSBhY3Rpb24gd2hlbiB0aGUgY29uZGl0"
    "aW9uYWwgbWVhbiBpcwogICAgICAgICAgICAjIHBvc2l0aXZlIChUMS9UNSkuIEl0IGlzIGEgRElS"
    "RUNUSU9OIGNoZWNrLCBuZXZlciBhIG1hZ25pdHVkZSBvciBhIHNoYXJlLgogICAgICAgICAgICBj"
    "b25zaXN0ZW50ID0gYm9vbCgocHMgPiAwKSA9PSAoY3VydmVfc2xvcGUgPiAwKSkKICAgICAgICBz"
    "bG9wZXMuYXBwZW5kKHsKICAgICAgICAgICAgImNvbnN0cmFpbnQiOiByWyJjb25zdHJhaW50Il0s"
    "ICJhcm0iOiByWyJhcm0iXSwKICAgICAgICAgICAgInBvbGljeV9hY3Rpb25fbGFnX3Nsb3BlIjog"
    "cHMsCiAgICAgICAgICAgICJwb2xpY3lfYWN0aW9uX2xhZ19zbG9wZV9zZCI6IGZsb2F0KHJbInNk"
    "X3Nsb3BlIl0pLAogICAgICAgICAgICAibGF3X2NvbmRpdGlvbmFsX21lYW5fbGFnX3Nsb3BlIjog"
    "Y3VydmVfc2xvcGUsCiAgICAgICAgICAgICJsYXdfc2xvcGVfbG8iOiBhbmFseXNpc1tsYXdfZm9y"
    "X2N1cnZlXVsiZXN0aW1hdGVzIl0uZ2V0KCJsYWdfc2xvcGVfbG8iKSwKICAgICAgICAgICAgImxh"
    "d19zbG9wZV9oaSI6IGFuYWx5c2lzW2xhd19mb3JfY3VydmVdWyJlc3RpbWF0ZXMiXS5nZXQoImxh"
    "Z19zbG9wZV9oaSIpLAogICAgICAgICAgICAiZGlyZWN0aW9uX2NvbnNpc3RlbnRfd2l0aF9sYXci"
    "OiBjb25zaXN0ZW50LAogICAgICAgIH0pCiAgICByZXR1cm4gewogICAgICAgICJsYXdfdXNlZF9m"
    "b3JfY3VydmUiOiBsYXdfZm9yX2N1cnZlLAogICAgICAgICJzb3VyY2VfdGFibGVzIjogeyJwb2xp"
    "Y3lfcmVzcG9uc2Vfc3VyZmFjZSI6IFNVUkZBQ0UsCiAgICAgICAgICAgICAgICAgICAgICAgICAg"
    "InBvbGljeV9yZXNwb25zZV9zbG9wZXMiOiBTTE9QRVN9LAogICAgICAgICJqb2luIjogInBvbGlj"
    "eSBsYWdnZWQtcmV0dXJuIGdyaWQgbWFwcGVkIHRvIGRpYWdub3N0aWMgeiBiaW5zIHZpYSB0aGUg"
    "ZnJvemVuICIKICAgICAgICAgICAgICAgICJlbXBpcmljYWwgTWVydG9uIG9uZS1zdGVwIGNhbGli"
    "cmF0aW9uIiwKICAgICAgICAicm93cyI6IHJvd3MsCiAgICAgICAgInNsb3BlX2NvbXBhcmlzb24i"
    "OiBzbG9wZXMsCiAgICAgICAgImludGVycHJldGF0aW9uX2xpbWl0IjoKICAgICAgICAgICAgIkRJ"
    "UkVDVElPTiBPTkxZLiBUaGlzIG92ZXJsYXkgc2hvd3Mgd2hldGhlciB0aGUgc2lnbiBvZiB0aGUg"
    "bGVhcm5lZCAiCiAgICAgICAgICAgICJhY3Rpb24gcmVzcG9uc2UgdG8gdGhlIGxhZ2dlZCByZXR1"
    "cm4gaXMgcXVhbGl0YXRpdmVseSBjb25zaXN0ZW50IHdpdGggdGhlICIKICAgICAgICAgICAgInNp"
    "Z24gb2YgdGhlIGNvbmRpdGlvbmFsIHN0cnVjdHVyZSBwcmVzZW50IGluIHRoZSBmcm96ZW4gdGFy"
    "Z2V0IGxhdy4gSXQgIgogICAgICAgICAgICAiZG9lcyBub3QgYXR0cmlidXRlIGFueSBzaGFyZSBv"
    "ZiB0aGUgcGVyZm9ybWFuY2UgZ2FwIHRvIHRoZSBsYWdnZWQtcmV0dXJuICIKICAgICAgICAgICAg"
    "ImNoYW5uZWwsIGFuZCBpdCBpcyBub3QgYSB0ZXN0LiIsCiAgICB9CgoKZGVmIHdyaXRlX292ZXJs"
    "YXkocGF0aCwgb3ZlcmxheSwgcnVuX21vZGUsIGV2aWRlbmNlX3Jvb3QpOgogICAgaW1wb3J0IGNv"
    "bXBhcmF0b3JfY29uZmlnIGFzIENDCiAgICBDQy5hc3NlcnRfbmFtZXNwYWNlX2lzb2xhdGlvbihw"
    "YXRoLCBydW5fbW9kZSwgZXZpZGVuY2Vfcm9vdCkKICAgIG9zLm1ha2VkaXJzKG9zLnBhdGguZGly"
    "bmFtZShwYXRoKSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIGZpZWxkcyA9IGxpc3Qob3ZlcmxheVsicm93"
    "cyJdWzBdLmtleXMoKSkKICAgIHdpdGggb3BlbihwYXRoLCAidyIsIG5ld2xpbmU9IiIpIGFzIGZo"
    "OgogICAgICAgIHcgPSBjc3YuRGljdFdyaXRlcihmaCwgZmllbGRuYW1lcz1maWVsZHMpCiAgICAg"
    "ICAgdy53cml0ZWhlYWRlcigpCiAgICAgICAgdy53cml0ZXJvd3Mob3ZlcmxheVsicm93cyJdKQog"
    "ICAgcmV0dXJuIHBhdGgK"
)
_SRC_OVERLAY = base64.b64decode(_SRC_OVERLAY_B64).decode()
open(os.path.join(SRC_DIR, "overlay.py"), "w").write(_SRC_OVERLAY)
print('overlay.py staged', len(_SRC_OVERLAY), 'chars')


## Module — static and unit checks with negative controls

Bin scheme, exact additivity, the Merton flatness positive control, an AR(1) negative control, a pairing-destruction negative control, resume exactness, the cluster unit, and an AST gate proving no actor training.


In [ ]:
_SRC_SMOKE_B64 = (
    "..."
)
_SRC_SMOKE = base64.b64decode(_SRC_SMOKE_B64).decode()
open(os.path.join(SRC_DIR, "smoke.py"), "w").write(_SRC_SMOKE)
print('smoke.py staged', len(_SRC_SMOKE), 'chars')


## Module — frozen namespace loader


In [ ]:
_SRC_FNS_B64 = (
    "IiIiTG9hZHMgdGhlIHZlcmlmaWVkIEJhc2UgMyBuYW1lc3BhY2UgaW4gQ29sYWIsIG1pcnJvcmlu"
    "ZyB0aGUgbG9jYWwgc2NyYXRjaCBsb2FkZXIuIiIiCmltcG9ydCBmcm96ZW5fbG9hZGVyIGFzIEZM"
    "CgoKZGVmIGxvYWRfZnJvemVuX25hbWVzcGFjZShyZXNvbHZlZCk6CiAgICBucyA9IEZMLmxvYWRf"
    "YmFzZTNfbmFtZXNwYWNlKHJlc29sdmVkWyJiYXNlM19ub3RlYm9vayJdKQogICAgRkwudmVyaWZ5"
    "X25hdGl2ZV9hc3RfaGFzaGVzKG5zKQogICAgcmV0dXJuIG5zCg=="
)
_SRC_FNS = base64.b64decode(_SRC_FNS_B64).decode()
open(os.path.join(SRC_DIR, "frozen_ns_stub.py"), "w").write(_SRC_FNS)
print('frozen_ns_stub.py staged')


## Step 00b — mode and hardware gate

`RESEARCH` raises here unless a CUDA NVIDIA T4 is allocated. There is no CPU fallback.


In [ ]:
# Step 00b — mode and hardware gate. RESEARCH stops here without a CUDA T4.
sys.path.insert(0, SRC_DIR) if SRC_DIR not in sys.path else None
import importlib
import comparator_config as CC
importlib.reload(CC)

PROFILE = CC.profile(RUN_MODE)
BACKEND, ENGINE_DEVICE, HARDWARE = CC.resolve_backend(
    RUN_MODE, allow_non_t4=ALLOW_NON_T4, allow_non_t4_reason=ALLOW_NON_T4_REASON)
OUT = CC.output_root(EVIDENCE_ROOT, RUN_MODE)
CC.write_json(os.path.join(OUT, "hardware_manifest.json"),
              {"hardware": HARDWARE, "backend": BACKEND,
               "engine_device": ENGINE_DEVICE, "profile": PROFILE,
               "work_dir": WORK, "inputs_dir": SP, "output_dir": OUT},
              RUN_MODE, "STEP_00_HARDWARE", EVIDENCE_ROOT)
print(json.dumps({"RUN_MODE": RUN_MODE, "backend": BACKEND,
                  "engine_device": ENGINE_DEVICE,
                  "device_name": HARDWARE.get("device_name"),
                  "is_t4": HARDWARE.get("is_t4"),
                  "evidence_class": PROFILE["evidence_class"],
                  "output_dir": OUT}, indent=2))
if RUN_MODE == "SMOKE":
    print("\n[SMOKE] Tiny budgets, CPU, isolated namespace. Nothing written by this "
          "run is scientific evidence.")
else:
    print("\n[RESEARCH] CUDA float32 batched engine on "
          f"{HARDWARE['device_name']} ({HARDWARE['total_memory_mb']} MB). "
          "Frozen Base 4 numerical contract preserved.")


## Run the diagnostic


In [ ]:
# Run the diagnostic.
#
# SMOKE  : 6 blocks x 64 paths per law; exercises every bin and the whole schema.
# RESEARCH: the bounded budget below, executed only after PMO authorisation.
#
# Resumable at block granularity: each (law, block) writes its own sufficient-statistic
# checkpoint, and a rerun recombines checkpoints instead of re-simulating. Pooling is in
# block-index order, so a resumed run reproduces an uninterrupted one exactly.

import importlib
for _m in ("binning", "laws", "runner", "overlay", "smoke"):
    importlib.invalidate_caches()

import binning as B, laws as L, runner as R, overlay as OV, smoke as S
import frozen_loader as FL, merton_arm as MA, comparator_config as CC
import frozen_ns_stub as FNS

EVIDENCE_ROOT = os.path.join(WORK, "evidence", "conditional_law_v1")
OUT_DIR = os.path.join(EVIDENCE_ROOT, "smoke" if RUN_MODE == "SMOKE" else "research")
os.makedirs(OUT_DIR, exist_ok=True)

ns = FNS.load_frozen_namespace(RESOLVED)
cal, train, meta = FL.build_frozen_environment(ns, RESOLVED["snapshot"])
rec = MA.calibrate_empirical_gbm(train, meta)
engine = FL.Base4Engine(ns, cal, backend=BACKEND, device=DEVICE)

prof = R.PROFILES[RUN_MODE]
print(json.dumps({"RUN_MODE": RUN_MODE, "profile": prof, "backend": BACKEND},
                 indent=1))

blocks, attempts, consts = R.simulate_blocks(
    ns, cal, rec, OUT_DIR, RUN_MODE, prof["n_blocks"], prof["n_paths"], engine=engine)
analysis = R.analyse(blocks, prof["n_boot"])

fp = {"base3_code_concat_sha256": ns["_base3_code_concat_sha256"],
      "snapshot_sha256": meta["snapshot_sha256"],
      "training_slice_sha256": meta["train_sha256"],
      "environment_fingerprint": ns["environment_fingerprint"](cal),
      "backend": BACKEND, "device": str(DEVICE)}
written, summary = R.write_outputs(OUT_DIR, analysis, attempts, consts, RUN_MODE,
                                   EVIDENCE_ROOT, fp, prof)

m1 = float(rec["m1_per_step_mean_log_increment"])
v1 = float(rec["v1_per_step_variance_log_increment"])
ov = OV.build_overlay(analysis, m1, v1)
written.append(OV.write_overlay(os.path.join(OUT_DIR, "policy_overlay.csv"), ov,
                                RUN_MODE, EVIDENCE_ROOT))

# The Merton flatness control is the positive control and runs in BOTH modes: if the
# iid arm does not come back flat, nothing else in this notebook should be believed.
control = S.s3_merton_flatness(ns, rec)
print("MERTON FLATNESS CONTROL:", json.dumps(
    {k: control[k] for k in ("pass", "lag_slope", "slope_interval_covers_zero",
                             "pooled_tail_probability")}, indent=1))
if not control["pass"]:
    raise RuntimeError("MERTON_IID_FLATNESS_CONTROL_FAILED")

for law in R.LAWS:
    e = analysis[law]["estimates"]
    print(f"{law:24s} slope {e['lag_slope']:+.5f} "
          f"[{e['lag_slope_lo']:+.5f}, {e['lag_slope_hi']:+.5f}]  "
          f"autocorr {e['lag1_autocorrelation']:+.5f}  n_pairs {e['n_pairs']:,}")
print("\nwrote:")
for p in written:
    print("  ", p)


## One-pass Colab instructions

1. Runtime → Change runtime type → **T4 GPU**. Run Step 00; it stops if the device is
   not a T4 and `ALLOW_NON_T4` is not set by a written PMO authorisation.
2. Run the module cells in order. They stage byte-identical copies of the modules that
   were smoke-executed; nothing is retyped.
3. Set `RUN_MODE = "RESEARCH"` **only after PMO has authorised the budget**, then run
   the final cell. If the session drops, re-run the same cell: completed blocks are
   reloaded from their checkpoints.
4. Upload the whole `evidence/conditional_law_v1/research/` folder to Drive and report
   the folder id, byte sizes and SHA-256 digests.

## What the outputs mean, and what they do not

`conditional_mean_bins.csv` is the key table: the conditional-mean response curve for
both laws on one common binning. Under the empirical Merton law the curve must be flat
up to Monte Carlo error — that is the built-in positive control, and the notebook
raises if it fails. Under the SBJTS law the curve may be state dependent.

A state-dependent SBJTS curve would show that lag-dependent conditional structure is
**present in the training law and absent from the comparator law by construction**.
That is a statement about the two market laws. It is **not** a causal decomposition of
the RL performance gap, and `policy_overlay.csv` is a direction-only juxtaposition:
it shows whether the sign of the already-accepted learned action response agrees with
the sign of the law's own conditional structure. No share, no attribution, no test.

Claim status remains `EXPLORATORY_MECHANISM_ONLY` whatever the numbers look like.
